# Kim et al. (2026), ICMS learning, tested on DANDI:001868

Paper. Kim, Lycke, Zolotavin, Montes, Xie and Luan (2026), "Learning induces
activation-mechanism-dependent neural plasticity in an intracortical microstimulation task",
bioRxiv doi:10.64898/2026.06.05.730421, version 1. The claims file (`claims/kim-2026-learning.yaml`)
was written from that preprint's full text (main text, legends, Methods); the supplement was not
supplied to its readers.

Dataset. DANDI:001868, version 0.260715.2016 (doi:10.48324/dandi.001868/0.260715.2016), streamed
from the archive. The paper cites no version of the dandiset; this is the only published version
(the draft has the same 85 assets and was last modified when it was published), so it is pinned.

What the files hold. 85 NWB files of 12 mice. The six behavioural mice (ICMS83, 92, 93, 98, 100,
101) have trials, ICMS trains, wheel position, sorted units (spike times, one mean waveform per unit,
a stored cell type) and volumetric dF/F per ROI, plus planar dF/F per ROI in the sessions with a
planar block. The six controls have per-ROI binary activation tables of a survey, and four of them
sorted units and stimulation tables. The tests run on the spikes, the trials and the wheel, and on
the controls' activation tables.

| Claim | Tested here |
|---|---|
| fig2a-threshold-decrease, figs1e-response-time, figs5-movement-onset | tested (behaviour) |
| fig4h-unit-counts, figs3-modulation-all-cells, fig4g-modulated-count-mouse1, fig4ij-pyramidal-drive | tested (ephys) |
| fig5d-pl-identity-stable, fig5e-pyramidal-in-pl, figs4b-conduction-velocity, fig5fg-pl-modulation-latency, figs4de-npl-no-change, fig5h-pl-spike-probability, figs5-pre-movement-window | tested (ephys, pulse-locking) |
| figs7-control-no-change, figs2ef-control-stable | tested (controls) |
| fig6c-kl-divergence, fig6d-npl-hit-miss-divergence, fig6f-pl-coupling-increase, fig6f-npl-higher-coupling, figs9-coupling-controls | tested (ephys and behaviour) |
| fig2b, fig2c, fig2d, fig2e (density and radius), fig2g | blocked: the volumetric dF/F samples cannot be placed in the pre-stimulus and stimulation windows |
| fig3b, fig3df, fig3e, fig3i, fig3j (two), fig3lm, fig3n (two), figs6 | blocked: no cross-session identity of the planar ROIs |
| fig4d-waveform-separation, figs8-tracked-units | blocked: per-spike or per-period waveforms are not in the files |

In [ ]:
import time, zlib, resource, warnings, hashlib, collections
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import mannwhitneyu, fisher_exact
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import icms

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
KEY = "kim-2026-learning"
OFFSETS = [0, 1000, 2000, 3000, 4000]


def seed(claim):
    """The claim's seed (docs/protocol.md, Author, step 3)."""
    return zlib.crc32(f"{KEY}/{claim}".encode())


def mw(a, b, alternative="two-sided"):
    """Mann-Whitney U of a against b with the rank-biserial r = 1 - 2U/(n1 n2) of the Python code
    (claims file header), U being a's statistic; returns (p, r, n_a, n_b). With a the late group, as
    called here, r is negative when late values are higher; its size is what compares with the paper."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    if len(a) == 0 or len(b) == 0:
        return np.nan, np.nan, len(a), len(b)
    u = mannwhitneyu(a, b, alternative=alternative)
    return u.pvalue, 1 - 2 * u.statistic / (len(a) * len(b)), len(a), len(b)


T0 = time.time()
print("dandiset", icms.DANDISET, "version", icms.VERSION)

In [ ]:
assets = icms.list_assets()
eph = assets[assets.kinds.str.contains("ecephys")]
with ThreadPoolExecutor(8) as ex:
    S = list(ex.map(lambda a: icms.load_session(a, wheel=True), eph.asset_id))
print(f"loaded {len(S)} ephys sessions in {time.time() - T0:.0f} s")

In [ ]:
# Weeks count days from the mouse's first session in the dandiset, of any kind (claims file header).
eph = assets[assets.kinds.str.contains("ecephys")]
first = assets.groupby("subject").session.min().to_dict()
first_ephys = eph.groupby("subject").session.min().to_dict()
for s in S:
    s["week"] = icms.week_of(s["session"], first[s["subject"]])
    s["phase"] = icms.phase_of(s["week"])
BEH = [s for s in S if s["subject"] in icms.BEHAVIOUR_MICE]
CTL = [s for s in S if s["subject"] in icms.CONTROL_MICE]

In [ ]:
print("Assets by subject and kind")
print(assets.groupby(["subject", "kinds"]).size().unstack(fill_value=0).to_string())
print("\nBehavioural sessions: week (days since first session, ceil / 7), trials, units")
rows = []
for s in BEH:
    tr = s["trials"]
    rows.append(dict(subject=s["subject"], mouse=icms.MOUSE_NUMBER[s["subject"]], session=s["session"],
                     week=s["week"], phase=s["phase"], trials=len(tr),
                     planar=int((tr.get("imaging_modality", pd.Series(dtype=str)) == "planar").sum()),
                     catch=int((tr.current_uA == 0).sum()), units=len(s["spikes"]),
                     currents=",".join(str(int(c)) for c in sorted(tr.current_uA.unique()) if c > 0),
                     channels=",".join(str(int(c)) for c in sorted(tr.stim_channel.unique()) if c > 0)))
SESS = pd.DataFrame(rows)
print(SESS.to_string())
print("\nsessions per mouse and phase:")
print(SESS.groupby(["subject", "phase"]).size().unstack(fill_value=0))
print("sessions in week 5 or later (dropped):", (SESS.week > 4).sum())
UNITS = pd.concat([s["units"].assign(subject=s["subject"], session=s["session"], week=s["week"]) for s in BEH])
UNITS["file_type"] = UNITS.cell_type.str.lower().map(
    lambda x: "pyramidal" if "pyr" in x else ("interneuron" if "inter" in x else "unknown"))
UNITS["t2p_type"] = np.where(UNITS.t2p_ms >= 0.45, "pyramidal", "interneuron")
print("\nunits in the 6 behavioural mice:", len(UNITS), "(paper 973)")
print("cell_type labels as stored:", UNITS.cell_type.value_counts().to_dict())
print("trough-to-peak of waveform_mean (ms): range %.3f to %.3f; agreement of t2p >= 0.45 ms with the stored label: %.3f"
      % (UNITS.t2p_ms.min(), UNITS.t2p_ms.max(),
         (UNITS.t2p_type == UNITS.file_type)[UNITS.file_type != "unknown"].mean()))
print(pd.crosstab(UNITS.file_type, UNITS.t2p_type))
print("waveform_mean shape per unit:", BEH[0]["waveforms"].shape[1:], "samples (one channel); unit 'volts', values %.3g to %.3g"
      % (min(s["waveforms"].min() for s in BEH), max(s["waveforms"].max() for s in BEH)))

# Recordings deposited twice: identical spike trains or identical stimulation tables.
h_sp, h_st = collections.defaultdict(list), collections.defaultdict(list)
for s in S:
    for i, sp in enumerate(s["spikes"]):
        if len(sp):
            h_sp[hashlib.md5(np.round(np.diff(sp), 6).tobytes()).hexdigest()].append((s["subject"], s["session"], i))
    if s["stim"] is not None:
        t = s["stim"].start_time.to_numpy()
        h_st[hashlib.md5(np.round(t - t[0], 4).tobytes()).hexdigest()].append((s["subject"], s["session"]))
print("\nidentical spike trains across units or sessions:", [v for v in h_sp.values() if len(v) > 1])
print("identical stimulation tables across sessions:", [v for v in h_st.values() if len(v) > 1])

print("\nStimulation table against trial table (ephys blocks):")
rows = []
for s in BEH:
    ev = icms.stim_events(s)
    m = ev.merge(s["trials"][["trial_index", "start_time"]], on="trial_index", suffixes=("", "_trial"))
    d = (m.start_time - m.start_time_trial) * 1000
    rows.append(dict(subject=s["subject"], session=s["session"], trains=len(ev),
                     dur_ms=f"{(ev.stop_time - ev.start_time).min() * 1000:.0f}-{(ev.stop_time - ev.start_time).max() * 1000:.0f}",
                     pulses=",".join(map(str, sorted(ev.pulse_count.unique()))),
                     freq=",".join(map(str, sorted(ev.frequency_hz.unique()))),
                     onset_minus_trial_ms=f"{d.min():.1f}/{d.median():.1f}/{d.max():.1f}",
                     stim_ch_in_electrodes=all(c in s["electrode_depth"] for c in ev.stim_channel.unique())))
STIMCHK = pd.DataFrame(rows)
print(STIMCHK.to_string())
print("\nresponse_time of hits (s): %.3f to %.3f" % (np.nanmin([s["trials"].response_time.min() for s in BEH]),
                                                       np.nanmax([s["trials"].response_time.max() for s in BEH])))
print("wheel: processed position at %s Hz, unit %s" % ({s["wheel_rate"] for s in BEH}, "a.u. (as stored)"))
print("\nControl sessions with ephys:")
for s in CTL:
    st = s["stim"]
    print(f"  {s['subject']} {s['session']} week {s['week']} (from first ephys: "
          f"{icms.week_of(s['session'], first_ephys[s['subject']])}) units {len(s['spikes'])} trials "
          f"{icms.control_trials(s).trial.nunique()} (trial_index values {st.trial_index.nunique()}) bursts {len(st)} pulses/burst {sorted(st.pulse_count.unique())} "
          f"freq {sorted(st.frequency_hz.unique())} currents {sorted(st.current_uA.unique())}")

### Control stimulation against the control spikes

The control stimulation tables list 3 bursts per trial, at 0, 0.5 and 1.0 s. Below, per session,
what the trial table gives (pulses and duration of each trial, recording blocks, trial_index);
the summed firing rate of all units around the first burst of each trial, in 0.25 s bins from -3
to 6 s; and, per current, the summed rate in 200 ms slots at 0.5 s spacing from the first burst
(k = 0 to 9) and in the 300 ms gaps after them.

In [ ]:
edges = np.arange(-3, 6.001, 0.25)
for s in CTL:
    tr = s["trials"]
    ct = icms.control_trials(s)
    print(f"{s['subject']} {s['session']}: trial table n_pulses {sorted(tr.n_pulses.unique())}, duration "
          f"{(tr.stop_time - tr.start_time).min():.2f} to {(tr.stop_time - tr.start_time).max():.2f} s, rep_num "
          f"{tr.rep_num.min()} to {tr.rep_num.max()}, trial_index odd {int((tr.trial_index % 2 == 1).sum())} even "
          f"{int((tr.trial_index % 2 == 0).sum())}, median spacing of trial starts {np.median(np.diff(np.sort(tr.start_time))):.2f} s; "
          f"stimulation table bursts per trial {sorted(ct.groupby('trial').size().unique())}")
    if not len(s["spikes"]):
        print("   no units"); continue
    t0 = ct.groupby("trial").start_time.min()
    cur = ct.groupby("trial").current_uA.first()
    allsp = np.sort(np.concatenate(s["spikes"]))
    h = sum(np.histogram(allsp - t, edges)[0] for t in t0) / len(t0) / 0.25
    last = ct.groupby("trial").stop_time.max().to_numpy() - t0.to_numpy()
    print(f"   0.25 s bins (last tabled burst ends {np.median(last):.2f} s after the first): " + " ".join(f"{x:.0f}" for x in h))
    for c in sorted(cur.unique()):
        tt = t0[cur == c].to_numpy()
        slot = [icms.window_counts(allsp, tt + 0.5 * k, 0, 0.2).mean() / 0.2 for k in range(10)]
        gap = [icms.window_counts(allsp, tt + 0.5 * k, 0.2, 0.5).mean() / 0.3 for k in range(10)]
        print(f"   {c:.0f} uA ({len(tt)} trials), 200 ms slots at k x 0.5 s: " + " ".join(f"{x:.0f}" for x in slot)
              + " | 300 ms gaps after them: " + " ".join(f"{x:.0f}" for x in gap))
print("0.25 s bins start at", edges[0], "s; 0 to 0.25 s is the bin of the first burst; rates in spikes/s summed over units")

### Clock check: spikes against the stimulation table

The trial table has no other stream to contrast with but the spikes. A unit that follows the
stimulation is modulated when its counts are taken at the true onsets, and not when the onsets are
moved into the inter-trial interval. Below, per session, the fraction of responses (4 to 6 uA)
that pass the modulation test at the true onsets and at onsets shifted by -2 s and +2 s (the
2.5 s post-stimulation period and the inter-trial interval of the Methods, Behavioral task). The
wheel is checked the same way: deviation within 700 ms of onset on hit against miss trials.

In [ ]:
rng_clock = np.random.default_rng(seed("clock-check"))
rows = []
for s in BEH + CTL:
    ev = icms.stim_events(s) if s["subject"] in icms.BEHAVIOUR_MICE else icms.control_trials(s).query("burst_index_in_trial == 0")
    ev = ev[ev.current_uA.isin([4.0, 5.0, 6.0])]
    if not len(ev) or not len(s["spikes"]):
        rows.append(dict(subject=s["subject"], session=s["session"], n=0)); continue
    out = dict(subject=s["subject"], session=s["session"], n=len(ev))
    for lab, sh in [("true", 0.0), ("minus2s", -2.0), ("plus2s", 2.0)]:
        fr = []
        for (ch, cur), g in ev.groupby(["stim_channel", "current_uA"]):
            st0 = g.start_time.to_numpy() + sh
            for spk in s["spikes"]:
                a = icms.window_counts(spk, st0, 0, 0.2); b = icms.window_counts(spk, st0, -0.2, 0)
                if len(a) > 2 and np.any(a != b):
                    t, p = stats.ttest_rel(a, b)
                    fr.append(p < 0.05 and t > 0)
                else:
                    fr.append(False)
        out[lab] = np.mean(fr)
    if "wheel" in s and s["subject"] in icms.BEHAVIOUR_MICE:
        w, fs, w0 = s["wheel"], s["wheel_rate"], s["wheel_t0"]
        dev = lambda g: np.median([np.abs(w[i:i + 700] - w[i]).max() for i in np.round((g.start_time.to_numpy() - w0) * fs).astype(int) if i + 700 < len(w)])
        out["wheel_hit"] = dev(ev[ev.is_hit == True]) if (ev.is_hit == True).any() else np.nan
        out["wheel_miss"] = dev(ev[ev.is_hit == False]) if (ev.is_hit == False).any() else np.nan
    rows.append(out)
CLOCK = pd.DataFrame(rows)
print(CLOCK.round(3).to_string())
print("\nsessions where the true onsets give a larger fraction than both shifts:",
      int(((CLOCK.true > CLOCK.minus2s) & (CLOCK.true > CLOCK.plus2s)).sum()), "of", int((CLOCK.n > 0).sum()))
wh = CLOCK.dropna(subset=["wheel_hit", "wheel_miss"])
print("behavioural sessions where the median wheel deviation is larger on hits than on misses:",
      int((wh.wheel_hit > wh.wheel_miss).sum()), "of", len(wh), "; the others:",
      wh[wh.wheel_hit <= wh.wheel_miss][["subject", "session", "wheel_hit", "wheel_miss"]].round(3).values.tolist())

### Volumetric imaging layout

The per-ROI volumetric dF/F is stored at about 30 Hz with NaN between trials. Below, for one session,
the number of samples per trial, their times relative to stimulation onset, and the median over
ROIs of each sample by current. The files do not say which samples are pre-stimulus and which are
in the stimulation window. Below also the full description of the series, a test of one reading
of it, and the columns of the segmentation tables of every behavioural file with imaging.

In [ ]:
a = assets[(assets.subject == "ICMS92") & (assets.session == "2023-09-06")].iloc[0]
f = icms.open_nwb(a.asset_id)
d = f["processing/ophys/DfOverF/DfOverF_Volumetric"]
rate = float(d["starting_time"].attrs["rate"])
X = d["data"][:]
print("volumetric dF/F", X.shape, "rate", round(rate, 3), "Hz;", d.attrs["description"][:200])
fin = np.isfinite(X[:, 0])
on = np.flatnonzero(np.diff(np.r_[0, fin.astype(int)]) == 1)
off = np.flatnonzero(np.diff(np.r_[fin.astype(int), 0]) == -1)
L = off - on + 1
print("samples per trial segment:", dict(zip(*np.unique(L, return_counts=True))))
trv = icms.behaviour_blocks([s for s in BEH if s["session"] == "2023-09-06" and s["subject"] == "ICMS92"][0])
tt = np.arange(len(X)) / rate
seg = []
for o, l in zip(on, L):
    j = np.argmin(np.abs(trv.start_time.to_numpy() - tt[o]))
    if l == 8 and abs(trv.start_time.iloc[j] - tt[o]) < 0.1:
        seg.append((trv.current_uA.iloc[j], tt[o:o + 8] - trv.start_time.iloc[j], np.median(X[o:o + 8], axis=1)))
print("sample times from onset (s), first trial:", np.round(seg[0][1], 3))
for c in sorted({x[0] for x in seg}):
    v = np.array([x[2] for x in seg if x[0] == c])
    print(f"  {c:.0f} uA, {len(v)} trials: median over ROIs per sample", np.round(np.median(v, 0), 3))
print("fraction of ROI samples equal to -1 per sample index:",
      np.round(np.mean(np.stack([X[o:o + 8] for o, l in zip(on, L) if l == 8]) == -1, axis=(0, 2)), 3))
print("\nfull description of DfOverF_Volumetric:", d.attrs["description"])
print("description of DfOverF_Planar:", f["processing/ophys/DfOverF/DfOverF_Planar"].attrs["description"])
# A reading of "pre/mid 4D stacks": the first four samples pre-stimulus, the last four in the
# stimulation window, with -1 taken as missing. Mid minus pre per ROI and trial, by current.
dif = collections.defaultdict(list)
for o, l in zip(on, L):
    j = np.argmin(np.abs(trv.start_time.to_numpy() - tt[o]))
    if l == 8 and abs(trv.start_time.iloc[j] - tt[o]) < 0.1:
        x = np.where(X[o:o + 8] == -1, np.nan, X[o:o + 8])
        dif[trv.current_uA.iloc[j]].append(np.nanmean(x[4:], 0) - np.nanmean(x[:4], 0))
print("reading 'first 4 pre, last 4 mid': median over trials of the ROI median of mid minus pre, and share of ROI-trials above 0:")
for c in sorted(dif):
    v = np.array(dif[c])
    print(f"  {c:.0f} uA ({len(v)} trials): {np.median(np.nanmedian(v, 1)):.3f}, {np.nanmean(v > 0):.3f}")
f.close()


def seg_columns(asset_id):
    g = icms.open_nwb(asset_id)
    p = "processing/ophys/"
    segs = [p + k + "/" + kk for k in g[p].keys() if k.startswith("ImageSegmentation") for kk in g[p + k].keys()]
    out = tuple(sorted((s[len(p):], tuple(sorted(g[s].keys()))) for s in segs))
    # What else the file holds that a registration of the planar sessions could use: the number of
    # planar ROIs, the members of each segmentation's reference_images, the groups under
    # processing/ophys and the acquisition group.
    inv = dict(n_planar=len(g[p + "ImageSegmentation_Planar/PlaneSegmentation_Planar/id"]) if "ImageSegmentation_Planar" in g[p] else None,
               refs=tuple(sorted(x for s in segs for x in g[s + "/reference_images"].keys())),
               ophys=tuple(sorted(g[p].keys())), acq=tuple(sorted(g["acquisition"].keys())))
    g.close()
    return out, inv


oph = assets[assets.subject.isin(icms.BEHAVIOUR_MICE) & assets.kinds.str.contains("ophys")]
with ThreadPoolExecutor(8) as ex:
    res = list(ex.map(seg_columns, oph.asset_id))
cols, inv = [x[0] for x in res], [x[1] for x in res]
print(f"\ncolumns of the segmentation tables in the {len(oph)} behavioural files with imaging:")
for k, n in collections.Counter(cols).items():
    print(f"  {n} files:", "; ".join(f"{t}: {', '.join(c)}" for t, c in k))
npl = [x["n_planar"] for x in inv if x["n_planar"] is not None]
print(f"planar ROIs per session, in the {len(npl)} files with a planar segmentation: {min(npl)} to {max(npl)}")
print("members of the segmentation tables' reference_images (files):", dict(collections.Counter(x["refs"] for x in inv)))
print("groups under processing/ophys (files):", dict(collections.Counter(x["ophys"] for x in inv)))
print("members of acquisition (files):", dict(collections.Counter(x["acq"] for x in inv)))

### Passive-control survey imaging

The control files hold a binary activation per ROI and stimulation condition, ROI voxel masks in
pixel and plane indices, and the positions of the 32 probe contacts in um.

In [ ]:
CS = []
for _, a in assets[assets.subject.isin(icms.CONTROL_MICE) & assets.kinds.str.contains("ophys")].iterrows():
    f = icms.open_nwb(a.asset_id)
    p = "processing/ophys/"
    act = f[p + "ImageSegmentation_ControlSurvey/PlaneSegmentation_ControlSurvey/activation"][:]
    sc = f[p + "stimulus_conditions"]
    cond = pd.DataFrame(dict(stim_channel=sc["stim_channel"][:].astype(str), current=sc["current_uA"][:]))
    pos = f[p + "stim_electrode_positions"]
    zero = int(((pos["x"][:] == 0) & (pos["y"][:] == 0) & (pos["z"][:] == 0)).sum())
    has_scale = "grid_spacing" in f[p + "ImageSegmentation_ControlSurvey/PlaneSegmentation_ControlSurvey/imaging_plane"]
    for j, r in cond.iterrows():
        CS.append(dict(subject=a.subject, session=a.session, week=icms.week_of(a.session, first[a.subject]),
                       stim_channel=r.stim_channel, current=r.current, n_rois=act.shape[0],
                       activated=int(act[:, j].sum()), contacts_at_origin=zero, grid_spacing=has_scale))
    f.close()
CS = pd.DataFrame(CS)
print(CS.groupby(["subject", "session", "week"]).agg(conditions=("current", "size"), rois=("n_rois", "first"),
      currents=("current", lambda x: sorted(set(x))), contacts_at_origin=("contacts_at_origin", "first"),
      pixel_size_given=("grid_spacing", "first")).to_string())
print("peak memory so far: %.2f GB" % (resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))

## Responses

A response is one unit at one stimulation channel and current in one session (Methods, Quantifying
neuromodulation metrics), at 4, 5 and 6 uA. The helper computes the modulation test, the modulation
score, the time to maximum firing rate, and the pulse-locking of each modulated response. The
pulse-locking draws random numbers (bootstrap and null). Each claim that uses the classes draws
them under its own seed and the seed plus 1000 to 4000 (pl_classes below); the first pass, under the
seed of fig5d-pl-identity-stable, gives the same values as that claim's own draw.

In [ ]:
def responses(sessions, rng, **kw):
    out, keep = [], {}
    for s in sessions:
        r, k = icms.session_responses(s, rng, keep_spikes=True, **kw)
        if not len(r):
            continue
        u = s["units"]
        r["week"], r["phase"] = s["week"], s["phase"]
        lab = u.cell_type.str.lower().map(lambda x: "pyramidal" if "pyr" in x else ("interneuron" if "inter" in x else "unknown"))
        r["file_type"] = lab.to_numpy()[r.unit]
        r["t2p_ms"] = u.t2p_ms.to_numpy()[r.unit]
        r["cell"] = np.where(r.t2p_ms >= 0.45, "pyramidal", "interneuron")
        r["unit_y"] = u.y_um.to_numpy()[r.unit]
        r["stim_y"] = r.stim_channel.map(s["electrode_depth"])
        out.append(r)
        for kk, v in k.items():
            keep[(s["subject"], s["session"]) + kk] = v
    return pd.concat(out, ignore_index=True), keep

In [ ]:
t = time.time()
R, KEEP = responses(BEH, np.random.default_rng(seed("fig5d-pl-identity-stable")), trial_counts=True)
print(f"responses: {len(R)} in {time.time() - t:.0f} s")

In [ ]:
# Selection filters of the code, which the claims file follows (header, "Ephys responses"): more than
# 5 trials, 0 < z below 100 (50 for Figure 5), stimulation channels used on at least 5 session days.
ch_days = pd.Series({k: len({x["session"] for x in BEH if x["subject"] == k[0] and k[1] in set(icms.stim_events(x).stim_channel)})
                     for k in R.groupby(["subject", "stim_channel"]).groups})
R["ch_days"] = [ch_days[(a, b)] for a, b in zip(R.subject, R.stim_channel)]


def sel(df, zmax=100, mod="modulated", ch=True):
    m = df[mod] & (df.n_trials > 5) & (df.z > 0) & (df.z < zmax)
    if ch:
        m &= df.ch_days >= 5
    return df[m]


PLC = {}


def pl_classes(claim, off=0):
    """Pulse-locking of every response that has one, drawn under the claim's seed plus off (each claim
    its own generator). The responses are taken in the order in which session_responses drew them,
    so fig5d-pl-identity-stable at offset 0 gives the values stored in R."""
    if (claim, off) not in PLC:
        rng = np.random.default_rng(seed(claim) + off)
        out = []
        for i, r in R[R.pl.notna()].iterrows():
            rel, tr = KEEP[(r.subject, r.session, r.stim_channel, r.current, r.unit)]
            pl, obs, _, pk, _ = icms.pulse_locking(icms.pulse_histogram(rel, tr, r.n_trials), r.n_trials, rng)
            out.append((i, pl, obs, pk))
        PLC[(claim, off)] = pd.DataFrame(out, columns=["i", "pl", "pli", "peak_prob"]).set_index("i")
    return PLC[(claim, off)]


def with_seed(df, claim, off=0):
    """The responses with the pulse-locking drawn under the claim's seed plus off."""
    d = df.copy()
    c = pl_classes(claim, off).loc[d.index]
    d["pl"], d["pli"], d["peak_prob"] = c.pl.to_numpy(bool), c.pli.to_numpy(float), c.peak_prob.to_numpy(float)
    return d


print("stimulation channels and their session days:")
print(ch_days.to_string())
print("responses", len(R), "modulated", int(R.modulated.sum()), "(one-sided variant", int(R.modulated_one.sum()), ")")
print("selected, z < 100:", len(sel(R)), " z < 50:", len(sel(R, 50)))
t = time.time()
chk = pl_classes("fig5d-pl-identity-stable", 0)
print("recomputed classes equal those stored in R:", bool((chk.pl.astype(bool) == R.loc[chk.index, "pl"].astype(bool)).all()),
      f"({time.time() - t:.0f} s per claim and seed)")
print("pulse-locked fraction of selected z < 50 under the seeds of fig5d-pl-identity-stable:",
      {o: round(with_seed(sel(R, 50), "fig5d-pl-identity-stable", o).pl.mean(), 3) for o in OFFSETS})

In [ ]:
# Why about 80% of the selected responses are pulse-locked (the paper's counts imply about 63%):
# where the peaks of the PL curves lie, how the stimulation-window spikes fall within the interpulse
# interval (the blanked 0 to 1.5 ms included), and how often the procedure calls a response
# pulse-locked when its spikes are placed uniformly in the valid window (its chance rate on these
# responses; the Methods' null is the 99th percentile, a nominal 1%). The uniform placement is drawn
# by the second generator of fig5d-pl-identity-stable.
d = sel(R, 50)
pk = d[d.pl == True].peak_ms
print("peak time of the PL curves (ms from the pulse), responses per 0.5 ms bin from 1.5 to 9.5 ms:",
      np.histogram(pk, np.arange(1.5, 9.51, 0.5))[0].tolist())
print("  median %.2f ms; share in the first valid bin (1.5 to 2 ms) %.3f" % (pk.median(), (pk < 2).mean()))
ph = np.concatenate([np.mod(KEEP[(r.subject, r.session, r.stim_channel, r.current, r.unit)][0], icms.IPI_S) * 1000
                     for _, r in d.iterrows()])
print(f"stimulation-window spikes of the selected responses: {len(ph)}; share within 0 to 1.5 ms of a pulse {(ph < 1.5).mean():.4f} "
      f"(0.15 if uniform), within 9.5 to 10 ms {(ph >= 9.5).mean():.4f} (0.05 if uniform)")
print("share of these spikes per 0.5 ms bin of the interpulse interval:", np.round(np.histogram(ph, np.arange(0, 10.01, 0.5))[0] / len(ph), 4).tolist())
rng_null = np.random.default_rng(seed("fig5d-pl-identity-stable") + 1)
fp = []
for _, r in d.iterrows():
    rel, tr = KEEP[(r.subject, r.session, r.stim_channel, r.current, r.unit)]
    rel_u = np.floor(rel / icms.IPI_S) * icms.IPI_S + rng_null.uniform(0.0015, 0.0095, len(rel))
    fp.append(icms.pulse_locking(icms.pulse_histogram(rel_u, tr, r.n_trials), r.n_trials, rng_null)[0])
print(f"chance rate: responses called pulse-locked with their spikes placed uniformly in the valid window {np.mean(fp):.3f} (n {len(fp)})")
d = d.assign(pl=d.pl.astype(bool))
print("pulse-locked share of the selected responses by mouse:", d.groupby("subject").pl.mean().round(3).to_dict())
print("and by current:", d.groupby("current").pl.mean().round(3).to_dict())

The pulse-locked share is not an artefact at the blanking edge or a matter of false positives: the
PL curves peak at a median of 3.75 ms after the pulse, with 3.2% in the first valid bin, and with
spikes placed uniformly in the valid window the procedure calls 2.4% of the 2744 responses
pulse-locked (above the nominal 1%, as a 99th percentile of 100 null sets is a noisy threshold, but
far below the 80.5% found). The sorted spike trains hold no spikes in the first 1.5 ms after a
pulse (share 0.0000, against 0.15 if uniform), as the blanking of the Methods expects. The share
varies by mouse, from 0.728 (ICMS93) to 0.98 (ICMS83).

## fig2a-threshold-decrease

> Behavioural detection thresholds decreased with learning; late-phase thresholds were significantly
> lower than early-phase thresholds, from a median of 5.5 uA to 4.2 uA.

Each block's threshold is the 50% point of a cumulative Gaussian fitted with psignifit 4.3 to the
hit proportions of a channel, with the block's catch-trial false-alarm rate applied under the two
readings of the claim; the session threshold is the mean of the block thresholds (Methods,
Analysis of behavioral data), and a session with a block without a finite threshold has none. Blocks are 100 trials of the ephys blocks by trial_index; ICMS83 has
no catch trials, so its sessions are fitted as one block with a false-alarm rate of 0.

In [ ]:
t = time.time()
TH = icms.session_thresholds(BEH)
print(f"{len(TH)} block fits in {time.time() - t:.0f} s; failed fits: {int(TH.threshold.isna().sum())}")

In [ ]:
wk = {(s["subject"], s["session"]): (s["week"], s["phase"]) for s in BEH}
def session_mean(v):
    """The mean of the block thresholds (Methods); none when a block has no finite threshold."""
    return v.mean() if np.isfinite(v).all() else np.nan


TS = TH.groupby(["subject", "session", "stim_channel", "reading"]).threshold.agg(session_mean).reset_index()
TS["week"] = [wk[(a, b)][0] for a, b in zip(TS.subject, TS.session)]
TS["phase"] = [wk[(a, b)][1] for a, b in zip(TS.subject, TS.session)]
print("block false-alarm rate: median %.2f, range %.2f to %.2f" % (TH.fa.median(), TH.fa.min(), TH.fa.max()))
fin = np.isfinite(TH.threshold)
print("block thresholds (uA), finite: range %.2f to %.2f; not finite (no 50%% point, false-alarm rate near or above"
      " the hit rates, or a failed fit): %d of %d" % (TH.threshold[fin].min(), TH.threshold[fin].max(), (~fin).sum(), len(TH)))
print("blocks with false-alarm rate >= 0.5:", int((TH[TH.reading == 'a'].fa >= 0.5).sum()))
# Session threshold: the mean of the four block thresholds (Methods); a session with a block without
# a finite threshold (a failed fit, NaN, or no 50% point, infinite) has none (main), or the mean of
# its finite blocks (variant).
print("blocks without a finite threshold by reading:", {rd: (int(TH[TH.reading == rd].threshold.isna().sum()), int(np.isinf(TH[TH.reading == rd].threshold).sum())) for rd in "ab"}, "(NaN, infinite)")
TSv = TH[fin].groupby(["subject", "session", "stim_channel", "reading"]).threshold.mean().reset_index()
for rd in "ab":
    d = TS[(TS.reading == rd) & TS.phase.notna()]
    e, l = d[d.phase == "early"].threshold, d[d.phase == "late"].threshold
    p, r, nl, ne = mw(l, e)
    print(f"\nreading ({rd}): early median {e.median():.2f} uA (n {ne}), late median {l.median():.2f} uA (n {nl}); "
          f"two-sided MWU p = {p:.3g}, r = {abs(r):.2f}")
    print("  per mouse (early / late medians, n):")
    for m, g in d.groupby("subject"):
        print(f"    {m} (Mouse {icms.MOUSE_NUMBER[m]}): {g[g.phase == 'early'].threshold.median():.2f} / "
              f"{g[g.phase == 'late'].threshold.median():.2f}  ({(g.phase == 'early').sum()}, {(g.phase == 'late').sum()})")
    dv = TSv[TSv.reading == rd].merge(TS[["subject", "session", "stim_channel", "reading", "phase"]].drop_duplicates(),
                                      on=["subject", "session", "stim_channel", "reading"])
    dv = dv[dv.phase.notna()]
    pv, rv, nlv, nev = mw(dv[dv.phase == "late"].threshold, dv[dv.phase == "early"].threshold)
    print(f"  variant, mean of finite blocks: early {dv[dv.phase == 'early'].threshold.median():.2f} (n {nev}), "
          f"late {dv[dv.phase == 'late'].threshold.median():.2f} (n {nlv}), p = {pv:.3g}")
    d2 = d[d.subject != "ICMS83"]
    p2, r2, nl2, ne2 = mw(d2[d2.phase == "late"].threshold, d2[d2.phase == "early"].threshold)
    print(f"  without ICMS83 (no catch trials, one block): early {d2[d2.phase == 'early'].threshold.median():.2f}, "
          f"late {d2[d2.phase == 'late'].threshold.median():.2f}, p = {p2:.3g} (n {ne2}, {nl2})")
# Reading (a) with psignifit's own threshold parameter, the midpoint of the sigmoid before the guess and
# lapse rates are applied, in place of the 50% point of the fitted curve.
TP = TH[TH.reading == "a"].groupby(["subject", "session", "stim_channel"]).threshold_param.agg(session_mean).reset_index()
TP["phase"] = [wk[(a, b)][1] for a, b in zip(TP.subject, TP.session)]
e, l = TP[TP.phase == "early"].threshold_param, TP[TP.phase == "late"].threshold_param
p, r, nl, ne = mw(l, e)
print(f"\nvariant, reading (a) with psignifit's threshold parameter: early median {e.median():.2f} uA (n {ne}), late {l.median():.2f} (n {nl}), p = {p:.3g}, r = {abs(r):.2f}")
fig, ax = plt.subplots(figsize=(5, 3))
d = TS[(TS.reading == "a") & TS.week.le(4) & np.isfinite(TS.threshold)]
ax.scatter(d.week + np.random.default_rng(seed("fig2a-threshold-decrease")).uniform(-0.2, 0.2, len(d)), d.threshold, s=8)
ax.set_xlabel("week"); ax.set_ylabel("session threshold (uA), reading (a)")
print("plotted range %.2f to %.2f uA, none outside the axes" % (d.threshold.min(), d.threshold.max()))
plt.show()

**Result: reproduced.** With the guess rate fixed at the catch-trial false-alarm rate (reading
a), the median session threshold is 5.70 uA early (n 43) and 3.91 uA late (n 80), the paper's
counts, both within tolerance of 5.5 and 4.2 uA, and lower late (p = 2.1e-07). With hit proportions
corrected for false alarms (reading b), 5.78 uA (n 46) and 4.07 uA (n 89), p = 9.97e-08, also within
tolerance. Thresholds are lower late in each of the six mice under both readings. With
psignifit's own threshold parameter under reading (a) the medians are 5.77 and 3.95 uA (p = 5e-08).

## figs1e-response-time

> Response times decreased over training with a time course like that of the detection thresholds.

Hit-trial response times at 4 to 6 uA, early against late, two-sided Mann-Whitney U; readings (a)
trials pooled, (b) one median per channel and session.

In [ ]:
rows = []
for s in BEH:
    ev = icms.stim_events(s)
    h = ev[(ev.is_hit == True) & ev.current_uA.isin(icms.CURRENTS)]
    rows.append(h.assign(subject=s["subject"], session=s["session"], week=s["week"], phase=s["phase"])
                [["subject", "session", "week", "phase", "stim_channel", "current_uA", "response_time"]])
RT = pd.concat(rows)
RT["rt_ms"] = RT.response_time * 1000
print("hit-trial response times (ms): range %.1f to %.1f, n = %d" % (RT.rt_ms.min(), RT.rt_ms.max(), len(RT)))
e, l = RT[RT.phase == "early"].rt_ms, RT[RT.phase == "late"].rt_ms
p, r, nl, ne = mw(l, e)
print(f"(a) trials: early median {e.median():.1f} ms (n {ne}), late {l.median():.1f} ms (n {nl}), p = {p:.3g}, r = {r:.2f}")
cs = RT[RT.phase.notna()].groupby(["subject", "session", "stim_channel", "phase"]).rt_ms.median().reset_index()
p, r, nl, ne = mw(cs[cs.phase == "late"].rt_ms, cs[cs.phase == "early"].rt_ms)
print(f"(b) channel-sessions: early median {cs[cs.phase == 'early'].rt_ms.median():.1f} ms (n {ne}), "
      f"late {cs[cs.phase == 'late'].rt_ms.median():.1f} (n {nl}), p = {p:.3g}")
print("by week (median ms, trials):", RT.groupby("week").rt_ms.agg(["median", "size"]).round(1).to_dict("index"))
print("per mouse, early / late median ms:")
for m, g in RT.groupby("subject"):
    print(f"  {m}: {g[g.phase == 'early'].rt_ms.median():.1f} / {g[g.phase == 'late'].rt_ms.median():.1f}, "
          f"p = {mw(g[g.phase == 'late'].rt_ms, g[g.phase == 'early'].rt_ms)[0]:.3g}")

**Result: reproduced.** Late hit-trial response times are shorter than early ones, 179.2 against
192.3 ms over trials (p = 2.83e-11) and 181.7 against 222.8 ms over channel-sessions (p = 0.00256).
Weekly medians fall from 231.6 ms in week 0 to 158.1 ms in week 4, not monotonically (198.5 ms in
week 2). Five mice are faster late; ICMS83 is slower late (356.0 against 254.9 ms).

## figs5-movement-onset

> Wheel-movement onset decreased with training; the median onset was 122 ms in the fastest mouse,
> 135 ms over all hit trials and 129 ms in the late phase, faster than early.

Onset is the first time the wheel deviates from its position at onset by more than half the
response threshold for 50 ms (claims file, from the code). The threshold in wheel units is not in
the files or the paper; it is estimated per session as the median deviation at the recorded
response time of hits (an assumption). The five mice with reliable encoders are not named; the
sanity check above shows that ICMS83's wheel barely moves at the recorded response times, so the
other five are used, and ICMS83 is shown apart. Thresholds 0.75 and 1.25 times the estimate are
variants.

In [ ]:
rows = []
for s in BEH:
    o = icms.movement_onsets(s)
    rows.append(o.assign(subject=s["subject"], session=s["session"], week=s["week"], phase=s["phase"]))
MO = pd.concat(rows)
print("estimated response threshold per session (wheel a.u.):",
      MO.groupby("subject").threshold.agg(["min", "median", "max"]).round(1).to_dict("index"))
print("hit trials after the end of the stored wheel trace:", MO.groupby(["subject", "session"]).beyond_wheel.sum().loc[lambda x: x > 0].to_dict())
MO = MO[~MO.beyond_wheel.astype(bool)]
print("hit trials without an onset within 700 ms:", MO.groupby("subject").onset_ms.apply(lambda x: int(x.isna().sum())).to_dict(),
      "of", MO.groupby("subject").size().to_dict())
for name, d0 in [("all currents", MO), ("4 to 6 uA", MO[MO.current.isin(icms.CURRENTS)])]:
    d = d0[(d0.subject != "ICMS83") & d0.onset_ms.notna()]
    per = d.groupby("subject").onset_ms.median()
    e, l = d[d.phase == "early"].onset_ms, d[d.phase == "late"].onset_ms
    p, r, nl, ne = mw(l, e)
    print(f"\n{name}, 5 mice: median onset per mouse (ms):", {f"{k} (Mouse {icms.MOUSE_NUMBER[k]})": round(v, 1) for k, v in per.items()})
    print(f"  fastest mouse {per.idxmin()} (Mouse {icms.MOUSE_NUMBER[per.idxmin()]}), {per.min():.1f} ms; ICMS92 (Mouse 1) {per.get('ICMS92', np.nan):.1f} ms")
    print(f"  all hit trials {d.onset_ms.median():.1f} ms (n {len(d)}); early {e.median():.1f} (n {ne}); late {l.median():.1f} (n {nl}); "
          f"two-sided MWU p = {p:.3g}, r = {r:.2f}")
    print("  onset range %.0f to %.0f ms; onset minus response time: median %.1f ms" % (d.onset_ms.min(), d.onset_ms.max(), (d.onset_ms - d.rt_ms).median()))
for sc in [0.75, 1.25]:
    MOs = pd.concat([icms.movement_onsets(s, scale=sc).assign(subject=s["subject"], phase=s["phase"]) for s in BEH])
    d = MOs[~MOs.beyond_wheel.astype(bool) & MOs.current.isin(icms.CURRENTS) & (MOs.subject != "ICMS83") & MOs.onset_ms.notna()]
    per = d.groupby("subject").onset_ms.median()
    p = mw(d[d.phase == "late"].onset_ms, d[d.phase == "early"].onset_ms)[0]
    print(f"variant, threshold x {sc}, 4 to 6 uA: fastest {per.idxmin()} {per.min():.1f} ms (ICMS92 {per.get('ICMS92', np.nan):.1f}); all hits {d.onset_ms.median():.1f}; "
          f"early {d[d.phase == 'early'].onset_ms.median():.1f}, late {d[d.phase == 'late'].onset_ms.median():.1f}, p = {p:.3g}")
d = MO[(MO.subject == "ICMS83") & MO.onset_ms.notna()]
print(f"\nICMS83 (not used): {len(d)} of {(MO.subject == 'ICMS83').sum()} hits with an onset, median {d.onset_ms.median():.1f} ms")

**Result: reproduced, resting on assumptions.** At 4 to 6 uA, the amplitudes the Methods analyse,
Mouse 1 (ICMS92) is the fastest of the five mice at 125.0 ms (122 +- 12.2), the median over hit
trials is 143.0 ms (135 +- 13.5) and the late median 135.0 ms (129 +- 12.9), faster than early
(155.0 ms, p = 6.3e-42). Over all currents the three medians (128.0, 148.0 and 137.0 ms) are also
within tolerance. The response threshold of the wheel is estimated from the data: at 0.75 and 1.25
times the estimate the three medians are 120.0, 134.0 and 127.0 ms, and 130.0, 151.0 and 142.0 ms,
so the absolute values follow the estimate (the last two outside tolerance at 1.25 times), while
Mouse 1 stays fastest and late onsets stay earlier (p = 1.8e-44 and 3.24e-38). ICMS83 is taken to be
the mouse without a reliable encoder; its deviation at the recorded response time is a median of
1.2 wheel units against 15.4 to 18.5 in the others.

## fig4h-unit-counts

> Of 973 recorded units (672 pyramidal, 301 interneurons), 484 were significantly modulated by the
> stimulation train (311 pyramidal, 173 interneurons), with a bimodal spike-width distribution in both.

Units are the sorted units of each session (not tracked). A unit is modulated when any of its
responses at 4 to 6 uA passes the modulation test with 0 < z < 100 (claims file). Cell type by the
trough-to-peak at 0.45 ms recomputed from the stored mean waveform, the criterion's rule (main, as
in fig4ij-pyramidal-drive and fig5e-pyramidal-in-pl), and by the stored cell_type label, which the
file describes as the putative cell type from t2p classification (variant). The pre-stimulus
windows of 300 and 500 ms are variants.

In [ ]:
# The modulation test with pre-stimulus windows of 300 and 500 ms (variants of the assumed 700 ms);
# no pulse-locking, so no random draws.
t = time.time()
RPRE = {}
for pre in [0.3, 0.5]:
    RPRE[pre], _ = responses(BEH, None, pre_s=pre, pulse=False)
    RPRE[pre]["ch_days"] = [ch_days[(a, b)] for a, b in zip(RPRE[pre].subject, RPRE[pre].stim_channel)]
print(f"pre-stimulus window variants in {time.time() - t:.0f} s")

In [ ]:
def unit_counts(M, label):
    mu = M.groupby(["subject", "session", "unit"]).size().index
    U = UNITS.reset_index(drop=True).copy()
    U["unit"] = U.groupby(["subject", "session"]).cumcount()
    U = U.set_index(["subject", "session", "unit"])
    mod = U.loc[mu]
    print(f"{label}: modulated units {len(mod)}; by t2p (main): {mod.t2p_type.value_counts().to_dict()}; "
          f"by stored label (variant): {mod.file_type.value_counts().to_dict()}")
    return U, mod


print("recorded units:", len(UNITS), "; by t2p >= 0.45 ms (main):", UNITS.t2p_type.value_counts().to_dict(),
      "; by stored label (variant):", UNITS.file_type.value_counts().to_dict())
U, MOD = unit_counts(sel(R), "main (two-sided paired t-test, z < 100, channels with >= 5 days)")
unit_counts(sel(R, ch=False), "variant, all stimulation channels")
unit_counts(sel(R, mod="modulated_one"), "variant, one-sided paired t-test")
unit_counts(R[R.modulated], "variant, the Methods' rule alone (no z, trial or channel filter)")
for pre in sorted(RPRE):
    unit_counts(sel(RPRE[pre]), f"variant, pre-stimulus window {pre * 1000:.0f} ms")
dis = UNITS[(UNITS.file_type != "unknown") & (UNITS.file_type != UNITS.t2p_type)]
print(f"units whose stored label and recomputed trough-to-peak disagree: {len(dis)}; recomputed t2p of those stored as pyramidal "
      f"{dis[dis.file_type == 'pyramidal'].t2p_ms.min():.3f} to {dis[dis.file_type == 'pyramidal'].t2p_ms.max():.3f} ms, of those stored as "
      f"interneurons {dis[dis.file_type == 'interneuron'].t2p_ms.min():.3f} to {dis[dis.file_type == 'interneuron'].t2p_ms.max():.3f} ms "
      f"(median {dis[dis.file_type == 'interneuron'].t2p_ms.median():.3f})")
print("per mouse, recorded / modulated:", {m: (int((UNITS.subject == m).sum()), int((MOD.reset_index().subject == m).sum())) for m in icms.BEHAVIOUR_MICE})
edges = np.arange(0.1, 1.25, 1 / 30)  # one sample per bin (30 kHz)
for name, d in [("all units", UNITS), ("modulated units", MOD)]:
    h = np.histogram(d.t2p_ms, edges)[0]
    c = (edges[:-1] + edges[1:]) / 2
    lo, hi = c < 0.45, c >= 0.45
    print(f"{name}: t2p histogram (1 sample bins from 0.1 ms):", h.tolist())
    print(f"   mode below 0.45 ms at {c[lo][np.argmax(h[lo])]:.3f} ms ({h[lo].max()} units), mode above at "
          f"{c[hi][np.argmax(h[hi])]:.3f} ms ({h[hi].max()}); minimum between them {h[(c > c[lo][np.argmax(h[lo])]) & (c < c[hi][np.argmax(h[hi])])].min()}")
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(UNITS.t2p_ms, edges, label="all"); ax.hist(MOD.t2p_ms, edges, label="modulated"); ax.axvline(0.45, color="k")
ax.set_xlabel("trough-to-peak (ms)"); ax.legend()
print("t2p range %.3f to %.3f ms; outside the axes: %d" % (UNITS.t2p_ms.min(), UNITS.t2p_ms.max(), int(((UNITS.t2p_ms < 0.1) | (UNITS.t2p_ms > 1.25)).sum())))
plt.show()

**Result: not reproduced.** The six mice hold 999 units (973 +- 97.3), 724 pyramidal and 275
interneurons by trough-to-peak (732 and 260 by the stored label), against 672 and 301. 588 units
are modulated (484 +- 48.4), 402 pyramidal and 186 interneurons (406 and 179 by the stored label),
against 311 and 173; with a one-sided test 599, and with pre-stimulus windows of 300 and 500 ms 653
and 633. Both distributions have a mode below 0.45 ms (0.383 ms) and one above (0.717 ms for all
units, 0.617 ms for modulated ones). The modulated total is outside its tolerance under every
window and cell-type rule shown, so the claim fails by its criterion whichever is taken.

## figs3-modulation-all-cells

> In representative longitudinal recordings, average modulation scores of all cells increased over
> training and activation latency decreased mildly; at session-specific detection thresholds the
> modulation score stayed stable.

Modulated responses (z < 100) pooled over the 6 mice; per current two-sided Mann-Whitney U, early
against late, of modulation score and time to maximum firing rate. At threshold: the response at
the current nearest each channel's session threshold, as in fig2g-at-threshold-stable, with the
session thresholds of fig2a-threshold-decrease under each reading, among the responses at 4 to 6
uA that the criterion sets. Main: the threshold rounded to the nearest integer current, kept when
that current is 4, 5 or 6 uA and was presented. Variant: the nearest of 4, 5 and 6 uA, so that a
threshold below 3.5 uA maps to 4 uA. Variant outside the criterion: the rounded threshold at every
current presented (1 to 14 uA).

In [ ]:
# Responses at every current presented, for the at-threshold test; no pulse-locking, so no random draws.
t = time.time()
RA, _ = responses(BEH, None, currents=tuple(float(c) for c in range(1, 15)), pulse=False)
RA["ch_days"] = [ch_days[(a, b)] for a, b in zip(RA.subject, RA.stim_channel)]
print(f"responses at all currents: {len(RA)}, modulated and selected {len(sel(RA))}, in {time.time() - t:.0f} s")

In [ ]:
M = sel(R)
res = {}
for c in icms.CURRENTS:
    d = M[M.current == c]
    e, l = d[d.phase == "early"], d[d.phase == "late"]
    pz, rz, _, _ = mw(l.z, e.z); pt, rt, _, _ = mw(l.ttm_ms, e.ttm_ms)
    res[c] = (pz, l.z.median() > e.z.median(), pt, l.ttm_ms.median() < e.ttm_ms.median())
    print(f"{c:.0f} uA: z early {e.z.median():.2f} (n {len(e)}), late {l.z.median():.2f} (n {len(l)}), p = {pz:.3g}, r = {rz:.2f}; "
          f"time to max early {e.ttm_ms.median():.0f} ms, late {l.ttm_ms.median():.0f} ms, p = {pt:.3g}, r = {rt:.2f}")
print("z range %.2f to %.2f; time to max range %.0f to %.0f ms" % (M.z.min(), M.z.max(), M.ttm_ms.min(), M.ttm_ms.max()))
MA = sel(RA)
READINGS = [("4 to 6 uA, threshold rounded to the nearest integer current (main)", M, lambda t: float(np.round(t))),
            ("4 to 6 uA, the nearest of 4, 5 and 6 uA (variant)", M, lambda t: float(np.clip(np.round(t), 4, 6))),
            ("every current presented, threshold rounded (variant)", MA, lambda t: float(np.round(t)))]
for label, Mx, nearest in READINGS:
    for rd in "ab":
        th = TS[TS.reading == rd].set_index(["subject", "session", "stim_channel"]).threshold
        near, n_thr, n_cur = [], 0, 0
        for (sub, ses, ch), t in th.items():
            if not np.isfinite(t) or wk[(sub, ses)][1] is None:
                continue
            n_thr += 1
            c = nearest(t)
            n_cur += bool(((RA.subject == sub) & (RA.session == ses) & (RA.stim_channel == ch) & (RA.current == c)).any())
            near.append(Mx[(Mx.subject == sub) & (Mx.session == ses) & (Mx.stim_channel == ch) & (Mx.current == c)])
        A = pd.concat(near)
        e, l = A[A.phase == "early"], A[A.phase == "late"]
        p, r, _, _ = mw(l.z, e.z)
        print(f"at threshold, {label}, reading ({rd}): channel-sessions with a threshold {n_thr}, whose current was presented {n_cur}; "
              f"responses by current {A.current.value_counts().sort_index().to_dict()}; z early {e.z.median():.2f} (n {len(e)}), "
              f"late {l.z.median():.2f} (n {len(l)}), p = {p:.3g}, r = {r:.2f}")
        res[(label, rd)] = p
print("per mouse, z early / late medians pooled over currents:",
      {m: (round(g[g.phase == 'early'].z.median(), 2), round(g[g.phase == 'late'].z.median(), 2)) for m, g in M.groupby("subject")})

**Result: partially reproduced.** Modulation scores are higher late at 4, 5 and 6 uA (p =
6.84e-05, 0.00853, 0.000404), and the time to maximum is shorter late at 5 uA (45 against 85 ms, p
= 0.00441). At the current nearest each session threshold within 4 to 6 uA the modulation score is
higher late, not stable: 2.53 against 4.13 (p = 2.59e-05) under threshold reading (a) and 2.64
against 4.21 (p = 4.96e-06) under (b) with the rounded threshold; 2.80 against 3.83 (p = 2.91e-05)
and 2.95 against 3.94 (p = 0.000139) with the nearest of 4, 5 and 6 uA. Two of the three parts hold
under both readings of the at-threshold current. Only with currents outside 4 to 6 uA is the
at-threshold modulation stable (2.85 against 2.99, p = 0.484; 2.95 against 3.25, p = 0.461), when
thresholds that round to 2 or 3 uA enter (52 and 235 responses under reading a).

## fig4g-modulated-count-mouse1

> The number of modulated units increased moderately in some mice, as in Mouse 1 at 4 uA, with no
> significant change at the group level.

One count of modulated units (z < 100) per session and stimulation channel at 4 uA; one-sided
Mann-Whitney U, early less than late (Figure 4G legend). Sessions and channels where 4 uA was
presented count, with zero when no unit was modulated. The pre-stimulus windows of 300 and 500 ms
are variants.

In [ ]:
def counts_4ua(Rx):
    """One count of modulated units (z < 100) per session and channel at 4 uA, zero where none."""
    M = sel(Rx)
    P = Rx[(Rx.current == 4.0) & (Rx.ch_days >= 5)][["subject", "session", "stim_channel", "phase"]].drop_duplicates()
    cnt = M[M.current == 4.0].groupby(["subject", "session", "stim_channel"]).size().rename("n").reset_index()
    P = P.merge(cnt, how="left").fillna({"n": 0})
    return P[P.phase.notna()]


P = counts_4ua(R)
for m in icms.BEHAVIOUR_MICE:
    d = P[P.subject == m]
    e, l = d[d.phase == "early"].n, d[d.phase == "late"].n
    p, r, nl, ne = mw(l, e, "greater") if len(e) and len(l) else (np.nan, np.nan, len(l), len(e))
    print(f"{m} (Mouse {icms.MOUSE_NUMBER[m]}): early counts {e.astype(int).tolist()} (median {e.median()}), late median {l.median()} "
          f"(n {ne}, {nl}); one-sided p = {p:.3g}, r = {r:.2f}")
e, l = P[P.phase == "early"].n, P[P.phase == "late"].n
p, r, nl, ne = mw(l, e, "greater")
print(f"\ngroup (a), pooled over 6 mice: early median {e.median()} (n {ne}), late {l.median()} (n {nl}), one-sided p = {p:.3g}, r = {r:.2f}")
p2 = mw(l, e)[0]
print(f"   two-sided p = {p2:.3g}")
for pre in sorted(RPRE):
    Pv = counts_4ua(RPRE[pre])
    d = Pv[Pv.subject == "ICMS92"]
    pm, rm, nlm, nem = mw(d[d.phase == "late"].n, d[d.phase == "early"].n, "greater")
    pg = mw(Pv[Pv.phase == "late"].n, Pv[Pv.phase == "early"].n, "greater")[0]
    print(f"variant, pre-stimulus window {pre * 1000:.0f} ms: Mouse 1 medians {d[d.phase == 'early'].n.median()} and {d[d.phase == 'late'].n.median()} "
          f"(n {nem}, {nlm}), one-sided p = {pm:.3g}; group one-sided p = {pg:.3g}")

**Result: not reproduced, resting on an assumption.** Mouse 1 (ICMS92) at 4 uA has 8 early and 21
late session-channel counts, as in the paper, with medians 12.0 and 14.0; the one-sided test gives p
= 0.0571, above 0.05 (paper 0.044). Pooled over the six mice there is no significant change (p =
0.0805), as the paper says. Counts rise significantly in ICMS101 (p = 0.00232) and ICMS83 (p =
0.0165). With pre-stimulus windows of 300 or 500 ms, Mouse 1 gives p = 0.014 and 0.0127 but the
pooled test also becomes significant (p = 0.0361 and 0.0375), which would make the claim partially
reproduced; the outcome rests on the 700 ms window.

## fig4ij-pyramidal-drive

> Increases in modulation strength and decreases in latency were driven mainly by pyramidal cells,
> while interneurons showed little significant change.

Modulated responses (z < 100) split by cell type; per current two-sided Mann-Whitney U, early
against late, of modulation score and time to maximum (Figure 4I, J legend). The code's one-sided
tests are a variant. Cell type by the trough-to-peak at 0.45 ms recomputed here from the 90-sample mean waveform, the
criterion's rule (main), and by the stored cell_type label, which the file describes as the
"putative cell type from t2p classification" (variant). The outcome rests on the criterion's rule;
until round 3 it rested on the stored label and was reproduced.

In [ ]:
FIJ = {}
for typ_col in ["cell", "file_type"]:
    print(f"cell type from {'trough-to-peak at 0.45 ms recomputed here (main, the criterion)' if typ_col == 'cell' else 'the stored label (variant)'}:")
    for typ in ["pyramidal", "interneuron"]:
        for c in icms.CURRENTS:
            d = sel(R)
            d = d[(d[typ_col] == typ) & (d.current == c)]
            e, l = d[d.phase == "early"], d[d.phase == "late"]
            pz, rz, _, _ = mw(l.z, e.z); pt, rt, _, _ = mw(l.ttm_ms, e.ttm_ms)
            pz1 = mw(l.z, e.z, "greater")[0]; pt1 = mw(l.ttm_ms, e.ttm_ms, "less")[0]
            FIJ[(typ_col, typ, c)] = dict(pz=pz, up=l.z.median() > e.z.median(), pt=pt, down=l.ttm_ms.median() < e.ttm_ms.median())
            print(f"  {typ:11s} {c:.0f} uA: n {len(e)}/{len(l)}; z {e.z.median():.2f} -> {l.z.median():.2f}, p = {pz:.3g}, r = {rz:.2f} "
                  f"(one-sided {pz1:.3g}); ttm {e.ttm_ms.median():.0f} -> {l.ttm_ms.median():.0f} ms, p = {pt:.3g}, r = {rt:.2f} (one-sided {pt1:.3g})")
print("paper's interneuron n early/late at 4, 5 and 6 uA: 69/254, 113/180, 122/86 (claims file)")
for typ_col in ["cell", "file_type"]:
    fj = lambda t, c: FIJ[(typ_col, t, c)]
    pm = all(fj("pyramidal", c)["pz"] < 0.05 and fj("pyramidal", c)["up"] for c in icms.CURRENTS)
    pt = all(fj("pyramidal", c)["pt"] < 0.05 and fj("pyramidal", c)["down"] for c in (5.0, 6.0))
    im = all(fj("interneuron", c)["pz"] >= 0.05 for c in icms.CURRENTS)
    print(f"criterion, {'trough-to-peak (main)' if typ_col == 'cell' else 'stored label (variant)'}: pyramidal modulation up at all three currents {pm}; "
          f"pyramidal latency down at 5 and 6 uA {pt}; interneuron modulation p >= 0.05 at all three {im}")

**Result: partially reproduced.** With cell types by the
trough-to-peak at 0.45 ms, pyramidal modulation rises at all three currents (p = 0.000143,
0.000435, 0.000492) and pyramidal time to maximum falls at 5 uA (45 to 25 ms, p = 0.000362) but
not significantly at 6 uA (85 to 35 ms, p = 0.0587); interneuron modulation does not change at any
current (p = 0.117, 0.706, 0.229). The pyramidal modulation result holds and one other does not.
With the stored label all three parts hold (pyramidal latency at 6 uA p = 0.025; interneuron
modulation p = 0.331, 0.884, 0.0885), which gave reproduced until round 3. The recomputed types are the closer to the paper's counts:
interneuron n early/late 63/228, 94/164 and 92/76, against 55/221, 90/158 and 90/76 by the stored
label and 69/254, 113/180 and 122/86 in the paper, and 724 pyramidal and 275 interneurons recorded,
against 732 and 260 and the paper's 672 and 301 (fig4h-unit-counts). The interneuron latency at 6
uA is longer late under the stored label (130 to 190 ms, p = 0.0335), as in the paper (r > 0), and
not significantly under trough-to-peak (135 to 155 ms, p = 0.419). The 97 units where the two
disagree are stored as pyramidal at a recomputed 0.333 to 0.433 ms, or as interneurons at 0.467 to
0.900 ms (median 0.533). The pyramidal latency test at 6 uA is close to the line and was run only
with the 700 ms pre-stimulus window.

## fig5d-pl-identity-stable

> Over 80% of units kept their pulse-locked or non-pulse-locked identity at a given stimulation
> contact as current increased.

Unit-channel pairs (animal, session, channel, unit) with modulated responses (z < 50) at two or
more of 4, 5 and 6 uA, each classed as PL at all currents, NPL at all, NPL to PL, PL to NPL, or
mixed (legend). Pulse-locking under the claim's seed and the four others.

In [ ]:
def identity(df):
    g = df.sort_values("current").groupby(["subject", "session", "stim_channel", "unit"]).pl.apply(list)
    g = g[g.apply(len) >= 2]
    def cls(v):
        if all(v): return "PL all"
        if not any(v): return "NPL all"
        if v == sorted(v): return "NPL to PL"
        if v == sorted(v, reverse=True): return "PL to NPL"
        return "mixed"
    return g.apply(cls).value_counts()


FIG5D = {}
for off in OFFSETS:
    c = identity(with_seed(sel(R, 50), "fig5d-pl-identity-stable", off))
    FIG5D[off] = (c.get("PL all", 0) + c.get("NPL all", 0)) / c.sum()
    if off == 0:
        print("pairs:", int(c.sum()), "(paper 729);", c.to_dict())
print("fraction keeping identity, by seed offset:", {k: round(v, 3) for k, v in FIG5D.items()})
c = identity(with_seed(sel(R, 50, ch=False), "fig5d-pl-identity-stable"))
print("variant, all channels: pairs", int(c.sum()), "fraction", round((c.get("PL all", 0) + c.get("NPL all", 0)) / c.sum(), 3))

**Result: reproduced.** Of 906 unit-channel pairs with modulated responses at two or more
currents (paper 729), 657 are PL at all currents and 103 NPL at all, 0.839 of pairs, above 0.8. The
fraction is 0.839 to 0.852 under the five seeds.

## fig5e-pyramidal-in-pl

> Pyramidal cells were more prevalent among pulse-locked than among non-pulse-locked units (76.2%
> against 49.8%).

Unit-level class by the two-step majority of the Methods (ties NPL), one row per animal, session
and unit, from the modulated responses (z < 50); cell type by the recomputed trough-to-peak at 0.45 ms,
the criterion's rule (main), and by the stored label (variant). Two-sided Fisher's exact test. PL classes under the claim's
seed and the four others.

In [ ]:
def unit_class(df):
    cur = df.groupby(["subject", "session", "unit", "current"]).pl.agg(lambda v: v.sum() > len(v) / 2)
    return cur.groupby(["subject", "session", "unit"]).agg(lambda v: v.sum() > len(v) / 2).rename("pl_unit")


FIG5E = {}
for off in OFFSETS:
    d = with_seed(sel(R, 50), "fig5e-pyramidal-in-pl", off)
    uc = unit_class(d).reset_index()
    info = d.drop_duplicates(["subject", "session", "unit"])[["subject", "session", "unit", "file_type", "cell"]]
    uc = uc.merge(info)
    for col in ["cell", "file_type"]:
        x = uc[uc[col] != "unknown"]
        tab = pd.crosstab(x.pl_unit, x[col] == "pyramidal")
        fp = 100 * x[x.pl_unit][col].eq("pyramidal").mean(); fn = 100 * x[~x.pl_unit][col].eq("pyramidal").mean()
        p = fisher_exact(tab.to_numpy())[1]
        FIG5E[(off, col)] = (fp, fn, p)
        if off == 0:
            print(f"{'stored label (variant)' if col == 'file_type' else 't2p recomputed (main)'}: PL units {int(x.pl_unit.sum())}, NPL units {int((~x.pl_unit).sum())}; "
                  f"pyramidal among PL {fp:.1f}%, among NPL {fn:.1f}%; Fisher p = {p:.3g}")
print("by seed offset (t2p recomputed, main):", {o: tuple(round(v, 3) for v in FIG5E[(o, 'cell')]) for o in OFFSETS})
print("by seed offset (stored label, variant): ", {o: tuple(round(v, 3) for v in FIG5E[(o, 'file_type')]) for o in OFFSETS})

**Result: partially reproduced.** 72.0% of 447 PL units are pyramidal (76.2 +- 7.62) and 56.4% of
140 NPL units (49.8 +- 4.98), and the difference is significant (Fisher p = 0.000819): pyramidal
cells are more common among PL units, as the paper says, but the NPL fraction is above its
tolerance, under all five seeds (55.9 to 57.8%). With the stored label the fractions are 73.4 and
57.1% (p = 0.000475), the NPL fraction also above tolerance under all five seeds (55.1 to 58.0%).

## figs4b-conduction-velocity

> Spike latency of pulse-locked responses increased linearly with distance from the stimulation
> site, giving conduction velocities of 0.2 to 0.27 m/s, while latency jitter did not vary with distance.

Distance is the absolute depth difference between the stimulation contact and the unit (file
electrode rel_y and unit_y_um); PL responses (z < 50) binned in 120 um bins from 60 um; per current
an OLS line through the bin medians of latency for bins centred from 120 to 840 um; velocity = 1 /
slope. Latency and jitter under the claim's two readings. Responses whose stimulation channel is not
in the electrodes table have no distance and are dropped. The first lines check the contact depths:
units whose peak channel is a stimulation channel should sit at a distance near 0. This does not
test that the stimulation table's stim_channel numbers the same contacts as channel_name. For that,
the next lines ask whether the modulation test passes more often near the stimulation contact under
the files' mapping than under the other assignments of a mouse's contact depths to its stimulation
channels; it can show the mapping only if responses fall off with distance. PL classes
under the claim's seed and the four others.

In [ ]:
C4B = "figs4b-conduction-velocity"
print("Stimulation contacts: the stimulation table's stim_channel and the electrodes table's channel_name are both")
print("described as the Ripple channel number. Per mouse, the depth (rel_y) of each stimulation channel, and the")
print("units whose peak channel is that channel, with their distance from it:")
for m in icms.BEHAVIOUR_MICE:
    ss = [x for x in BEH if x["subject"] == m]
    chs = sorted({int(c) for x in ss for c in icms.stim_events(x).stim_channel.unique()})
    depth = {c: sorted({x["electrode_depth"].get(c, np.nan) for x in ss}) for c in chs}
    on = []
    for x in ss:
        u = x["units"]
        if "peak_channel" in u:
            for c in chs:
                if c in x["electrode_depth"]:
                    on += list(np.abs(u[u.peak_channel == c].y_um - x["electrode_depth"][c]))
    off_peak = np.concatenate([np.abs(x["units"].y_um - x["units"].peak_channel.map(x["electrode_depth"])) for x in ss if "peak_channel" in x["units"]])
    print(f"  {m}: stimulation channel -> depth (um) {depth}; units on a stimulation channel {len(on)}, distance median "
          f"{np.median(on) if on else np.nan:.1f}, max {np.max(on) if on else np.nan:.1f} um; all units, |unit_y - depth of peak channel| median {np.nanmedian(off_peak):.1f} um")
# The check above compares units with their own peak channel; it does not test that stim_channel
# numbers the same contacts as channel_name. If it does, units near the stimulation contact should
# respond more often. Spearman rho between distance and the modulation test's result over all
# responses (4 to 6 uA, more than 5 trials), under the files' mapping and under every other
# assignment of a mouse's stimulation-contact depths to its stimulation channels.
import itertools
RHO = {}
for m in icms.BEHAVIOUR_MICE:
    d = R[(R.subject == m) & R.stim_y.notna() & R.unit_y.notna() & (R.n_trials > 5)]
    chs = sorted(d.stim_channel.unique())
    dep = d.groupby("stim_channel").stim_y.first()
    rh = {}
    for perm in itertools.permutations(chs):
        mp = dict(zip(chs, dep.loc[list(perm)].to_numpy()))
        rh[perm] = stats.spearmanr((d.stim_channel.map(mp) - d.unit_y).abs(), d.modulated)[0]
    RHO[m] = (rh[tuple(chs)], np.array([v for k, v in rh.items() if k != tuple(chs)]))
    print(f"  {m}: channels {chs}, responses {len(d)}, modulated {d.modulated.mean():.2f}; rho under the files' mapping {RHO[m][0]:.3f}; "
          f"under the {len(RHO[m][1])} other assignments {np.round(np.sort(RHO[m][1]), 3).tolist()}")
tot = sum(v[0] for v in RHO.values())
alt = np.array([sum(x) for x in itertools.product(*[np.r_[v[0], v[1]] for v in RHO.values()])])
print(f"sum of rho over mice: files' mapping {tot:.3f}; fraction of the {len(alt)} joint assignments with a sum at or below it: {np.mean(alt <= tot + 1e-12):.4f}")
FIG4B = {}
for off in OFFSETS:
    d = with_seed(sel(R, 50), C4B, off)
    d = d[d.pl & d.stim_y.notna() & d.unit_y.notna()].copy()
    d["dist"] = (d.stim_y - d.unit_y).abs()
    if off == 0:
        print(f"PL responses with a distance: {len(d)}; dropped without a stimulation-contact depth: "
              f"{int((with_seed(sel(R, 50), C4B).pl & sel(R, 50).stim_y.isna()).sum())}; distance range {d.dist.min():.0f} to {d.dist.max():.0f} um")
    for rd, lat, jit in [("a", "lat_a", "jit_a"), ("b", "lat_b", "jit_b")]:
        for c in icms.CURRENTS:
            x = d[d.current == c]
            b = np.floor((x.dist - 60) / 120).astype(int)
            centre = 120 + 120 * b
            med = x.groupby(centre)[lat].median()
            nb = x.groupby(centre).size()
            m = med[(med.index >= 120) & (med.index <= 840)]
            fit = stats.linregress(m.index.to_numpy(float), m.to_numpy())
            v = 1 / fit.slope / 1000 if fit.slope > 1e-9 else np.nan  # um/ms -> m/s; a slope zero up to rounding gives none
            jf = stats.linregress(x.dist, x[jit])
            FIG4B[(off, rd, c)] = (v, jf.pvalue, fit.slope)
            if off == 0:
                print(f"({rd}) {c:.0f} uA: n {len(x)}; bin medians {m.round(2).to_dict()} (n per bin {nb.loc[m.index].tolist()}); "
                      f"slope {fit.slope * 1000:.3f} ms/mm (p {fit.pvalue:.3g}), velocity {v:.3f} m/s; jitter slope p = {jf.pvalue:.3g}")
print("velocities by seed offset:", {k: round(v[0], 3) for k, v in FIG4B.items()})

**Result: not reproduced.** Latency does not increase with distance: the bin-median slopes are
-0.577, -0.203 and 0.129 ms/mm at 4, 5 and 6 uA under reading (a) and -1.042, -0.298 and 0.000
under reading (b), so no velocity can be derived except at 6 uA under (a) (7.750 m/s), far above
0.18 to 0.297 m/s; no seed gives a velocity in that range. Jitter does not vary with distance
(slope p = 0.185 to 0.701), as the paper says. Medians of the first-spike latency lie between 4.24
and 5.12 ms in every bin. The contact mapping is not established by the data. Units sit a median
of 1.1 to 8.1 um (per mouse) from their own peak channel, and units whose peak channel is a
stimulation channel 5.3 to 24.4 um from it (none in ICMS92), which checks the depths of
channel_name, not the numbering of stim_channel. Whether a response is modulated does not fall with
distance under the files' mapping (Spearman rho -0.134 to 0.108 per mouse, summed 0.011, with 45%
of the other joint assignments at or below it), so no assignment stands out and a scrambled one is
not ruled out. The mapping rests on the files' description of both columns as the Ripple channel
number.

## fig5fg-pl-modulation-latency

> Pulse-locked responses increased their modulation from early to late at all currents and
> decreased latency significantly at 5 and 6 uA.

PL modulated responses, z < 50, more than 50 spikes; per current two-sided Mann-Whitney U, early
against late, of modulation score and time to maximum. PL classes under the claim's seed and the
four others.

In [ ]:
def pl_tests(d, label, pl=True, show=True):
    out = {}
    for c in icms.CURRENTS:
        x = d[(d.pl == pl) & (d.current == c) & (d.n_spikes > 50)]
        e, l = x[x.phase == "early"], x[x.phase == "late"]
        pz, rz, _, _ = mw(l.z, e.z); pt, rt, _, _ = mw(l.ttm_ms, e.ttm_ms); pp, rp, _, _ = mw(l.peak_prob, e.peak_prob)
        out[c] = dict(pz=pz, up=l.z.median() > e.z.median(), pt=pt, down=l.ttm_ms.median() < e.ttm_ms.median(),
                      pp=pp, pup=l.peak_prob.median() > e.peak_prob.median(), n=(len(e), len(l)))
        if show:
            print(f"  {label} {c:.0f} uA: n {len(e)}/{len(l)}; z {e.z.median():.2f} -> {l.z.median():.2f}, p = {pz:.3g}, r = {rz:.2f}; "
                  f"ttm {e.ttm_ms.median():.0f} -> {l.ttm_ms.median():.0f} ms, p = {pt:.3g}, r = {rt:.2f}; "
                  f"peak prob {e.peak_prob.median():.3f} -> {l.peak_prob.median():.3f}, p = {pp:.3g}, r = {rp:.2f}")
    return out


PLT = {off: pl_tests(with_seed(sel(R, 50), "fig5fg-pl-modulation-latency", off), "PL", True, off == 0) for off in OFFSETS}
for off in OFFSETS:
    o = PLT[off]
    print(f"seed +{off}: z p", [f"{o[c]['pz']:.3g}" for c in icms.CURRENTS], "ttm p", [f"{o[c]['pt']:.3g}" for c in icms.CURRENTS])

**Result: partially reproduced (seed-sensitive).** PL modulation rises at all three currents (p =
0.000272, 0.0017, 3.11e-05). The time to maximum falls at 5 uA (75 to 35 ms, p = 0.0032) but not
significantly at 6 uA (95 to 55 ms, p = 0.0741). Under the seed plus 1000 the 6 uA latency gives p =
0.0441, which would make the claim reproduced; the other three seeds give 0.0852 to 0.0968.

## figs4de-npl-no-change

> Non-pulse-locked responses showed no significant change in modulation or latency from early to late.

A claim of absence: the test of Figure 5F, G on NPL responses, per current, two-sided
Mann-Whitney U at p < 0.05, no correction. PL classes under the claim's seed and the four others.

In [ ]:
NPLT = {off: pl_tests(with_seed(sel(R, 50), "figs4de-npl-no-change", off), "NPL", False, off == 0) for off in OFFSETS}
for off in OFFSETS:
    o = NPLT[off]
    sig = sum((o[c]["pz"] < 0.05) + (o[c]["pt"] < 0.05) for c in icms.CURRENTS)
    print(f"seed +{off}: significant tests {sig} of 6; z p", [f"{o[c]['pz']:.3g}" for c in icms.CURRENTS],
          "ttm p", [f"{o[c]['pt']:.3g}" for c in icms.CURRENTS])

**Result: reproduced (seed-sensitive).** Under the claim's seed none of the six NPL tests is
significant (modulation p = 0.0893, 0.643, 0.856; time to maximum p = 0.188, 0.51, 0.808). Under the
seeds plus 2000 and plus 3000 the 4 uA modulation test gives p = 0.0245 and 0.0342, as the PL and
NPL classes of a few responses change.

## fig5h-pl-spike-probability

> Peak spike probability within the interpulse interval of pulse-locked responses increased over
> time at 5 and 6 uA.

Peak of the bootstrapped mean spike-probability curve of each PL response (z < 50, more than 50
spikes, as for Figure 5F, G), per current two-sided Mann-Whitney U, early against late. PL classes
under the claim's seed and the four others.

In [ ]:
C5H = "fig5h-pl-spike-probability"
PH = {off: pl_tests(with_seed(sel(R, 50), C5H, off), "PL", True, off == 0) for off in OFFSETS}
for off in OFFSETS:
    o = PH[off]
    print(f"seed +{off}: peak spike probability p", {c: f"{o[c]['pp']:.3g} ({'up' if o[c]['pup'] else 'down'})" for c in icms.CURRENTS})
d = with_seed(sel(R, 50), C5H, 0)
print("peak spike probability range (spikes per bin per trial, summed over pulses): %.3f to %.3f"
      % (d[d.pl].peak_prob.min(), d[d.pl].peak_prob.max()))

**Result: partially reproduced.** Peak spike probability of PL responses is higher late at 6 uA
(2.508 to 3.105, p = 0.0193) but not at 5 uA (p = 0.626); at 4 uA it does not change (p = 0.836), as
in the paper. All five seeds agree (6 uA p 0.00602 to 0.0193, 5 uA p 0.54 to 0.758).

## figs5-pre-movement-window

> Within a 120 ms window from stimulation onset, before movement, PL and NPL modulation changes were
> consistent with those in the 700 ms window.

Pre window 120 ms before and stimulation window 120 ms after the first pulse, z < 100, time to
maximum at 2 ms bins with sigma 5 ms (claims file, from the code). Readings for which responses
enter: (ii) those passing the modulation test in the 120 ms windows (main: the code recomputes the
responses in these windows, and its filters keep the modulated responses of the table they are
given); (i) those modulated in the 700 ms analysis, with z recomputed in the 120 ms windows
(variant). The PL classes are those of the 700 ms responses, drawn under the claim's seed and the
four others; the 120 ms run draws no random numbers.

In [ ]:
R120, _ = responses(BEH, None, stim_s=0.12, pre_s=0.12, ttm_bin=0.002, ttm_sigma=0.005, pulse=False)

In [ ]:
key = ["subject", "session", "stim_channel", "current", "unit"]
C5P = "figs5-pre-movement-window"
FIG5P = {}
for off in OFFSETS:
    Rc = with_seed(R[R.modulated], C5P, off)
    for name, d in [("(ii) modulated at 120 ms", R120[R120.modulated].merge(Rc[key + ["pl"]], on=key)),
                    ("(i) modulated at 700 ms", R120.merge(Rc[Rc.index.isin(sel(R).index)][key + ["pl"]], on=key))]:
        d["ch_days"] = [ch_days[(a, b)] for a, b in zip(d.subject, d.stim_channel)]
        d = d[(d.n_trials > 5) & (d.z > 0) & (d.z < 100) & (d.ch_days >= 5)]
        if off == 0:
            print(name, f"responses {len(d)}")
        for pl in [True, False]:
            for c in icms.CURRENTS:
                x = d[(d.pl == pl) & (d.current == c)]
                e, l = x[x.phase == "early"], x[x.phase == "late"]
                p, r, _, _ = mw(l.z, e.z); pt, rt, _, _ = mw(l.ttm_ms, e.ttm_ms)
                FIG5P[(off, name[:4].strip(), pl, c)] = (p, l.z.median() > e.z.median())
                if off == 0:
                    print(f"  {'PL ' if pl else 'NPL'} {c:.0f} uA: n {len(e)}/{len(l)}; z {e.z.median():.2f} -> {l.z.median():.2f}, p = {p:.3g}, r = {r:.2f}; "
                          f"ttm {e.ttm_ms.median():.0f} -> {l.ttm_ms.median():.0f} ms, p = {pt:.3g}")
for off in OFFSETS:
    for rd in ["(ii)", "(i)"]:
        plup = sum(FIG5P[(off, rd, True, c)][0] < 0.05 and FIG5P[(off, rd, True, c)][1] for c in icms.CURRENTS)
        nsig = [f"{FIG5P[(off, rd, False, c)][0]:.3g}" for c in icms.CURRENTS]
        print(f"seed +{off} {rd}: PL modulation higher late at {plup} of 3 currents; NPL p {nsig}")

**Result: partially reproduced, resting on an assumption.** With the responses that pass the
modulation test in the 120 ms windows (reading ii, adopted as the code recomputes the responses in
these windows), PL modulation is higher late at 4, 5 and 6 uA (p = 0.00033, 3.91e-05, 0.0107), but
NPL modulation is also higher late at 4 uA (p = 0.0321) and 6 uA (p = 0.000311), on 13 to 76
responses per group, which the 700 ms window does not show; all five seeds give the same pattern.
With the responses of the 700 ms analysis measured in 120 ms windows (reading i), PL modulation
rises at all three currents (p = 0.000143, 9.56e-05, 0.0011) and NPL at none (p = 0.089, 0.562,
0.468), which would make the claim reproduced.

## figs7-control-no-change

> In control mice without behavioural learning, neither PL nor NPL cells showed significant changes
> in modulation, latency or spike probability.

The control mice with recordings (ICMS45, 48, 54, 56) received 3 bursts of 10 pulses at 50 Hz per
trial (file metadata), not the 700 ms train; the response windows and pulse-locking window for these
trials are assumptions stated in the helper (control_responses). Of 4 to 6 uA only 5 uA was given.
Weeks by ceil(days / 7) from the mouse's first session in the dandiset; the stimulation-channel
filter is not applied, as each control channel was used in few sessions. Per measure, two-sided
Mann-Whitney U, early against late, for PL and NPL responses (z < 50, more than 50 spikes). The
spikes follow stimulation every 0.5 s for 5 s (sanity check), so a variant takes each trial as 10
bursts at 0.5 s spacing from the first tabled one, with a 2 s pre-stimulus window.

In [ ]:
RC, KEEPC, RC10, KEEPC10 = [], {}, [], {}
rngc = np.random.default_rng(seed("figs7-control-no-change"))
rngc10 = np.random.default_rng(seed("figs7-control-no-change") + 1)
for s in CTL:
    if not len(s["spikes"]):
        continue
    r, k = icms.control_responses(s, rngc, currents=(2.0, 5.0, 7.0, 10.0), keep_spikes=True)
    r10, k10 = icms.control_responses(s, rngc10, currents=(2.0, 5.0, 7.0, 10.0), n_bursts=10, keep_spikes=True)
    for x in (r, r10):
        x["week"], x["phase"] = s["week"], s["phase"]
        x["week_e"] = icms.week_of(s["session"], first_ephys[s["subject"]])
    RC.append(r); RC10.append(r10)
    for kk, v in k.items():
        KEEPC[(s["subject"], s["session"]) + kk] = v
    for kk, v in k10.items():
        KEEPC10[(s["subject"], s["session"]) + kk] = v
RC = pd.concat(RC, ignore_index=True)
RC10 = pd.concat(RC10, ignore_index=True)
for x in (RC, RC10):
    x["phase_e"] = x.week_e.map(icms.phase_of)

In [ ]:
print("control responses:", len(RC), "modulated:", int(RC.modulated.sum()))
print(RC[RC.modulated].groupby(["subject", "session", "week", "phase"]).size().to_string())
FIG7 = []
for trains, RCx in [("3 tabled bursts", RC), ("10 bursts at 0.5 s (variant)", RC10)]:
    C = RCx[RCx.modulated & (RCx.n_trials > 5) & (RCx.z > 0) & (RCx.z < 50) & (RCx.n_spikes > 50)]
    for ph in ["phase", "phase_e"]:
        for curs in [(5.0,), (2.0, 5.0, 7.0, 10.0)]:
            sig = 0
            for pl in [True, False]:
                x = C[(C.pl == pl) & C.current.isin(curs)]
                e, l = x[x[ph] == "early"], x[x[ph] == "late"]
                for m in ["z", "ttm_ms", "peak_prob"]:
                    p, r, nl, ne = mw(l[m], e[m])
                    sig += p < 0.05 if np.isfinite(p) else 0
                    FIG7.append(dict(trains=trains, weeks=ph, currents=curs, cls="PL" if pl else "NPL", measure=m, n_early=ne, n_late=nl,
                                     early=e[m].median(), late=l[m].median(), p=p, r=r))
            print(f"{trains}, weeks from {'first session' if ph == 'phase' else 'first ephys session'}, currents {curs}: significant tests {sig} of 6")
F7 = pd.DataFrame(FIG7)
print(F7[F7.weeks == "phase"].round(3).to_string())
print("weeks from the first ephys session give the same groups:",
      bool((F7[F7.weeks == "phase"][["n_early", "n_late"]].to_numpy() == F7[F7.weeks == "phase_e"][["n_early", "n_late"]].to_numpy()).all()))

**Result: not reproduced, resting on assumptions.** With the 3 tabled bursts, in the control mice
at 5 uA the peak spike probability of PL responses is lower in the late weeks (1.060 against 0.654,
p = 0.001), while their modulation (p = 0.972) and time to maximum (p = 0.294) do not change. One
NPL response is modulated early and one late, so the NPL tests have no power (p = 1.000). Taking
each trial as the 10 bursts at 0.5 s that the spikes follow (variant), no test at 5 uA is
significant (smallest p 0.076), with 20/8 PL and 19/24 NPL responses; over all control currents
NPL modulation then rises (2.10 to 3.32, p < 0.001). The outcome rests on the windows taken for
these trains.

## fig6c-kl-divergence

> Hit-miss separation of delta-spike distributions, by KL divergence, increased over time for both
> PL and NPL responses, and NPL responses separated hit from miss more than PL responses in both phases.

Delta spikes per trial = spikes in the 700 ms train minus spikes in the 700 ms before, for each
trial of each PL or NPL response (z < 50); trials pooled per week and class across mice; KL(hit ||
miss) by bootstrap as in the helper (kl_bootstrap), under the claim's two readings; bootstrap values
of weeks 0 and 1 against weeks 2 to 4 per class, and NPL against PL per phase, by two-sided
Mann-Whitney U with BH-FDR over the four tests. The PL classes are drawn under the claim's seed,
the bootstrap under its second generator.

In [ ]:
FIG6C = {}
for off in OFFSETS:
    D = with_seed(sel(R, 50), "fig6c-kl-divergence", off)
    D = D[D.hit.notna()]
    rng = np.random.default_rng(seed("fig6c-kl-divergence") + 1 + off)
    for rd, wins, prop in [("a", 0.01, 0.4), ("b", 0.025, 0.8)]:
        kl = {}
        for pl in [True, False]:
            for w in range(5):
                x = D[(D.pl == pl) & (D.week == w)]
                if not len(x):
                    continue
                dl = np.concatenate([a - b for a, b in zip(x.c_stim, x.c_pre)])
                hit = np.concatenate(list(x.hit))
                kl[(pl, w)] = icms.kl_bootstrap(dl[hit].astype(float), dl[~hit].astype(float), rng, prop=prop, wins=wins)
                if off == 0 and rd == "a":
                    print(f"  {'PL ' if pl else 'NPL'} week {w}: trials hit {hit.sum()}, miss {(~hit).sum()}, median KL {np.median(kl[(pl, w)]):.3f}")
        ph = lambda pl, ws: np.concatenate([kl[(pl, w)] for w in ws if (pl, w) in kl])
        tests = [("PL late vs early", ph(True, [2, 3, 4]), ph(True, [0, 1])),
                 ("NPL late vs early", ph(False, [2, 3, 4]), ph(False, [0, 1])),
                 ("NPL vs PL early", ph(False, [0, 1]), ph(True, [0, 1])),
                 ("NPL vs PL late", ph(False, [2, 3, 4]), ph(True, [2, 3, 4]))]
        ps = [mw(a, b)[0] for _, a, b in tests]
        q = multipletests(ps, method="fdr_bh")[1]
        FIG6C[(off, rd)] = {n: (np.median(a), np.median(b), qq, mw(a, b)[1]) for (n, a, b), qq in zip(tests, q)}
        if off == 0:
            print(f"reading ({rd}):")
            for n, (ma, mb, qq, r) in FIG6C[(off, rd)].items():
                print(f"   {n}: median KL {ma:.3f} vs {mb:.3f}, p_FDR = {qq:.3g}, r = {r:.2f}")
for k, v in FIG6C.items():
    print("seed +%d reading (%s):" % k, {n: ("higher" if a > b else "lower", f"{q:.2g}") for n, (a, b, q, _) in v.items()})

**Result: partially reproduced.** NPL responses separate hits from misses more than PL responses
in both phases under both readings and all five seeds (median KL 0.362 against 0.034 early and
0.368 against 0.027 late under reading a, p_FDR printed as 0). PL separation is lower late than early
under both readings and all seeds (0.027 against 0.034, p_FDR = 1.25e-123), the reverse of the paper.
The NPL change has no stable direction: higher late under reading (a) (0.368 against 0.362, p_FDR =
7.4e-06) and lower under (b) (0.194 against 0.219, p_FDR = 0.000475) at the claim's seed, and either
direction under the other seeds.

## fig6d-npl-hit-miss-divergence

> PL responses showed no significant hit-miss divergence at any time, while NPL responses diverged
> significantly from 65 ms after ICMS onset in the early phase and from 35 ms in the late phase.

Per response (unit, channel, current, session; z < 50) with at least 3 hit and 3 miss trials, the
trial-median firing rate over 0 to 700 ms for hits and for misses. Each trial's binned rate is
smoothed with a Gaussian of sigma 20 ms before the median, as a median of raw 5 or 10 ms counts is
zero for most units (an assumption; the smoothing of the code's time-to-maximum is used). Cluster
test as in the helper (cluster_test), bins of 5 and 10 ms (the claim's readings). Onset is the start
of the first significant cluster (main) and of the last (the code's choice, variant). A variant
without smoothing takes the trial mean of the raw 10 ms counts. The PL classes are drawn under
the claim's seed, the label swaps under its second generator.

In [ ]:
from scipy.ndimage import gaussian_filter1d
C6D = "fig6d-npl-hit-miss-divergence"
D = sel(R, 50)
D = D[D.hit.notna()]


def traces(bin_s, smooth=True, stat=np.median):
    """Per response with at least 3 hit and 3 miss trials: (index, phase, hit trace, miss trace)."""
    nb = int(round(0.7 / bin_s))
    out = []
    for i, r in D.iterrows():
        hit = r.hit
        if hit.sum() < 3 or (~hit).sum() < 3:
            continue
        rel, tr = KEEP[(r.subject, r.session, r.stim_channel, r.current, r.unit)]
        H = np.zeros((r.n_trials, nb))
        np.add.at(H, (tr, np.clip((rel / bin_s).astype(int), 0, nb - 1)), 1)
        H = H / bin_s
        if smooth:
            H = gaussian_filter1d(H, 0.020 / bin_s, axis=1, mode="nearest")
        out.append((i, r.phase, stat(H[hit], 0), stat(H[~hit], 0)))
    return out


FIG6D = {}
for label, bin_s, smooth, stat, offs in [("main", 0.010, True, np.median, OFFSETS), ("main", 0.005, True, np.median, [0]),
                                          ("raw mean", 0.010, False, np.mean, [0])]:
    T = traces(bin_s, smooth, stat)
    for off in offs:
        cls = with_seed(D, C6D, off).pl
        rng = np.random.default_rng(seed(C6D) + 1 + off)
        for pl in [True, False]:
            for ph in ["early", "late"]:
                sub = [t for t in T if cls[t[0]] == pl and t[1] == ph]
                h = np.array([t[2] for t in sub]); m = np.array([t[3] for t in sub])
                cl = icms.cluster_test(h, m, rng)
                sig = [c for c in cl if c[3] < 0.05]
                first = sig[0][0] * bin_s * 1000 if sig else np.nan
                last = sig[-1][0] * bin_s * 1000 if sig else np.nan
                FIG6D[(label, bin_s, off, pl, ph)] = (len(sub), len(sig), first, last)
                if off == 0:
                    print(f"{label}, bin {bin_s * 1000:.0f} ms, {'PL ' if pl else 'NPL'} {ph}: units {len(sub)}; clusters {len(cl)}, significant {len(sig)}: "
                          + ", ".join(f"{a * bin_s * 1000:.0f}-{(b + 1) * bin_s * 1000:.0f} ms (mass {ms:.1f}, p {p:.3f})" for a, b, ms, p in sig))
for k, v in FIG6D.items():
    print("%s, bin %.0f ms seed +%d %s %s:" % (k[0], k[1] * 1000, k[2], "PL" if k[3] else "NPL", k[4]),
          "significant clusters %d, first onset %s ms, last %s ms" % (v[1], v[2], v[3]))

**Result: not reproduced, resting on an assumption.** PL responses diverge between hits and misses
in significant clusters in both phases (from 340 ms early and 580 ms late, 10 ms bins), where the
paper finds none. NPL responses diverge from 180 ms early and from 70 ms late (5 ms bins: 185 and
65 ms), against 65 and 35 ms; the onset is earlier late, as in the paper. The NPL onsets are the
same under all five seeds; the clusters are not: the early PL cluster starts at 330 ms under three
seeds and 340 ms under two, the late PL phase has 2 clusters under the claim's seed and 1 under the
other four, and the early NPL phase has 2 to 4 clusters. PL clusters are present in both phases
under every seed. Without smoothing (trial means of raw 10 ms counts) PL clusters remain in both
phases and the NPL onsets are 100 and 60 ms, still outside tolerance.

## fig6f-pl-coupling-increase

> Population coupling of PL units increased from early to late, while NPL coupling stayed stable.

Population coupling as in the Methods (helper population_coupling): per session and condition
(channel and current), the spikes inside the trains concatenated over trials in 1 ms bins; the
population is every modulated response of that condition except the unit; PL and NPL units are the
responses with z < 50. Values pooled over 4 to 6 uA; two-sided Mann-Whitney U with BH-FDR over the
four tests of the panel. The PL classes are drawn under the claim's seed and the circular shifts under its second generator
(seed plus 1), each also plus 1000 to 4000.

In [ ]:
def coupling(keep, Rsel, Rpop, rng, window=0.7, pop_pl_only=False, bursts=False):
    """Normalised coupling of each selected response; Rsel the responses evaluated, Rpop those that
    form the population. With bursts, the concatenation is over the 200 ms bursts of the controls."""
    out = []
    span = 0.2 if bursts else window
    nb_per = int(round(span * 1000))
    popkeys = set(zip(Rpop.subject, Rpop.session, Rpop.stim_channel, Rpop.current, Rpop.unit))
    plkeys = set(zip(Rsel[Rsel.pl == True].subject, Rsel[Rsel.pl == True].session, Rsel[Rsel.pl == True].stim_channel,
                     Rsel[Rsel.pl == True].current, Rsel[Rsel.pl == True].unit))
    selkeys = set(zip(Rsel.subject, Rsel.session, Rsel.stim_channel, Rsel.current, Rsel.unit))
    for (sub, ses, ch, cur), g in Rpop.groupby(["subject", "session", "stim_channel", "current"]):
        trains = {}
        n_tr = None
        for u in g.unit:
            k = (sub, ses, ch, cur, u)
            rel, tr = keep[k]
            m = rel < window
            trains[k] = np.sort((np.floor(rel[m] * 1000).astype(int) + tr[m] * nb_per))
        if bursts:  # the number of bursts of the condition, from the burst indices of its spikes
            n_tr = 1 + max((int(keep[(sub, ses, ch, cur, u)][1].max()) for u in g.unit
                            if len(keep[(sub, ses, ch, cur, u)][1])), default=0)
        else:
            n_tr = int(g.n_trials.iloc[0])
        n_bins = n_tr * nb_per
        pop = {k for k in trains if (k in plkeys)} if pop_pl_only else None
        pc = icms.population_coupling(trains, n_bins, rng, pop_units=pop)
        for k, v in pc.items():
            if k in selkeys:
                out.append(dict(subject=k[0], session=k[1], stim_channel=k[2], current=k[3], unit=k[4], pc=v))
    out = pd.DataFrame(out)
    return out.merge(Rsel[["subject", "session", "stim_channel", "current", "unit", "pl", "phase"]])


def coupling_tests(C, label, show=True):
    pl, npl = C[C.pl == True], C[C.pl == False]
    tests = [("PL late vs early", pl[pl.phase == "late"].pc, pl[pl.phase == "early"].pc),
             ("NPL late vs early", npl[npl.phase == "late"].pc, npl[npl.phase == "early"].pc),
             ("NPL vs PL early", npl[npl.phase == "early"].pc, pl[pl.phase == "early"].pc),
             ("NPL vs PL late", npl[npl.phase == "late"].pc, pl[pl.phase == "late"].pc)]
    ps = [mw(a, b)[0] for _, a, b in tests]
    q = multipletests(ps, method="fdr_bh")[1]
    out = {}
    for (n, a, b), qq in zip(tests, q):
        p, r, na, nb = mw(a, b)
        out[n] = dict(a=np.median(a), b=np.median(b), q=qq, r=r, na=na, nb=nb)
        if show:
            print(f"  {label} {n}: medians {np.median(a):.3f} vs {np.median(b):.3f} (n {na}, {nb}), p_FDR = {qq:.3g}, r = {r:.2f}")
    return out

In [ ]:
PC = {}
C6F = "fig6f-pl-coupling-increase"
for off in OFFSETS:
    PC[off] = coupling(KEEP, with_seed(sel(R, 50), C6F, off), R[R.modulated], np.random.default_rng(seed(C6F) + 1 + off))

In [ ]:
print("units with fewer than 50 spikes in the trains are left out; evaluated:", len(PC[0]), "of", len(sel(R, 50)))
print("normalised coupling range %.2f to %.2f" % (PC[0].pc.min(), PC[0].pc.max()))
FIG6F = {off: coupling_tests(PC[off], "seed +%d" % off, off == 0) for off in OFFSETS}
for off in OFFSETS[1:]:
    print(f"seed +{off}:", {k: f"{v['q']:.3g} ({'higher' if v['a'] > v['b'] else 'lower'})" for k, v in FIG6F[off].items()})
print("per mouse, PL early / late medians:", {m: (round(g[(g.pl == True) & (g.phase == 'early')].pc.median(), 3), round(g[(g.pl == True) & (g.phase == 'late')].pc.median(), 3)) for m, g in PC[0].groupby("subject")})

**Result: reproduced (seed-sensitive).** PL coupling rises from early to late (median 0.272 to
0.707, p_FDR = 0.00159), while NPL coupling does not change significantly (1.750 to 1.535, p_FDR =
0.158). The PL rise holds under all five seeds (p_FDR 0.000478 to 0.00336); the NPL fall reaches
p_FDR = 0.0445 and 0.0456 under the seeds plus 2000 and plus 3000, which would make the claim
partially reproduced.

## fig6f-npl-higher-coupling

> NPL units had significantly higher population coupling than PL units in both early and late phases.

The tests of fig6f-pl-coupling-increase, with the PL classes and the coupling shifts drawn under
this claim's seed (classes) and its second generator (shifts).

In [ ]:
C6N = "fig6f-npl-higher-coupling"
FIG6FN = {}
for off in OFFSETS:
    pcn = coupling(KEEP, with_seed(sel(R, 50), C6N, off), R[R.modulated], np.random.default_rng(seed(C6N) + 1 + off))
    FIG6FN[off] = coupling_tests(pcn, "", False)
for off in OFFSETS:
    o = FIG6FN[off]
    print(f"seed +{off}: early NPL {o['NPL vs PL early']['a']:.3f} vs PL {o['NPL vs PL early']['b']:.3f} (n {o['NPL vs PL early']['na']}, {o['NPL vs PL early']['nb']}), "
          f"p_FDR = {o['NPL vs PL early']['q']:.3g}, r = {o['NPL vs PL early']['r']:.2f}; "
          f"late NPL {o['NPL vs PL late']['a']:.3f} vs PL {o['NPL vs PL late']['b']:.3f} (n {o['NPL vs PL late']['na']}, {o['NPL vs PL late']['nb']}), "
          f"p_FDR = {o['NPL vs PL late']['q']:.3g}, r = {o['NPL vs PL late']['r']:.2f}")

**Result: reproduced.** Under the claim's own classes and shifts NPL coupling is higher than PL
coupling early (1.698 against 0.285, p_FDR = 1.54e-11) and late (1.474 against 0.690, p_FDR =
1.54e-11), and under all five seeds (p_FDR at most 9.69e-11 early and 6.67e-10 late).

## figs9-coupling-controls

> The PL coupling increase was driven mainly by coupling among PL cells; the NPL above PL coupling
> difference held within the 120 ms pre-movement window and was absent in control mice.

(A) the population built from PL units only; (B) only spikes in the first 120 ms of each train; (C)
the control mice at 5 uA, concatenating the 200 ms bursts (the control trains, an assumption stated
in figs7-control-no-change). Each with two-sided Mann-Whitney U and BH-FDR over four tests. The PL classes of the trained mice are
drawn under the claim's seed and the coupling shifts under its second generator; the control classes
are those of figs7-control-no-change. A variant of (C) takes each control trial as 10 bursts at
0.5 s, with the classes of that variant in figs7-control-no-change.

In [ ]:
FIG9 = {}
C9 = "figs9-coupling-controls"
for off in OFFSETS:
    rng = np.random.default_rng(seed(C9) + 1 + off)
    show = off == 0
    Rs = with_seed(sel(R, 50), C9, off)
    A = coupling(KEEP, Rs, R[R.modulated], rng, pop_pl_only=True)
    B = coupling(KEEP, Rs, R[R.modulated], rng, window=0.12)
    Csel = RC[RC.modulated & (RC.n_trials > 5) & (RC.z > 0) & (RC.z < 50) & (RC.current == 5.0)]
    Cc = coupling(KEEPC, Csel, RC[RC.modulated & (RC.current == 5.0)], rng, window=0.2, bursts=True)
    if show:
        print(f"(A) PL-to-PL: evaluated {len(A)}")
    ta = coupling_tests(A, "(A)", show)
    if show:
        print(f"(B) first 120 ms: evaluated {len(B)}")
    tb = coupling_tests(B, "(B)", show)
    if show:
        print(f"(C) controls 5 uA: evaluated {len(Cc)}, PL {int((Cc.pl == True).sum())}, NPL {int((Cc.pl == False).sum())}")
    tc = coupling_tests(Cc, "(C)", show) if len(Cc) else {}
    # Variant: the control trials as 10 bursts at 0.5 s (figs7-control-no-change), with that variant's classes.
    Csel10 = RC10[RC10.modulated & (RC10.n_trials > 5) & (RC10.z > 0) & (RC10.z < 50) & (RC10.current == 5.0)]
    Cc10 = coupling(KEEPC10, Csel10, RC10[RC10.modulated & (RC10.current == 5.0)], rng, window=0.2, bursts=True)
    if show:
        print(f"(C, 10 bursts, variant) controls 5 uA: evaluated {len(Cc10)}, PL {int((Cc10.pl == True).sum())}, NPL {int((Cc10.pl == False).sum())}")
    tc10 = coupling_tests(Cc10, "(C, 10 bursts)", show) if len(Cc10) else {}
    FIG9[off] = (ta, tb, tc, tc10)
for off in OFFSETS:
    ta, tb, tc, tc10 = FIG9[off]
    print(f"seed +{off}: (A) PL late vs early p_FDR {ta['PL late vs early']['q']:.3g} ({'up' if ta['PL late vs early']['a'] > ta['PL late vs early']['b'] else 'down'}); "
          f"(B) NPL>PL early {tb['NPL vs PL early']['q']:.3g}, late {tb['NPL vs PL late']['q']:.3g}; "
          f"(C) NPL vs PL early {tc.get('NPL vs PL early', {}).get('q', np.nan):.3g}, late {tc.get('NPL vs PL late', {}).get('q', np.nan):.3g}; "
          f"(C, 10 bursts) NPL vs PL early {tc10.get('NPL vs PL early', {}).get('q', np.nan):.3g}, late {tc10.get('NPL vs PL late', {}).get('q', np.nan):.3g}")

**Result: partially reproduced, resting on an assumption.** With the population built from PL
units only, PL coupling rises from early to late (p_FDR = 0.00128; 0.000418 to 0.00246 under the
five seeds). In the first 120 ms of the trains NPL coupling exceeds PL coupling early (p_FDR =
2.81e-06) and late (p_FDR = 3.76e-08), under all five seeds. With the 3 tabled bursts the control
test cannot be made: at 5 uA the controls have 47 PL and 1 NPL coupling values, so the NPL against
PL comparison is undefined, and two of the three parts hold. Taking each control trial as 10 bursts
at 0.5 s, there are 22 PL and 40 NPL values and NPL against PL gives p_FDR = 0.924 early and 0.355
late (0.924 to 0.975 and 0.355 to 0.4 under the five seeds), so the absence holds and the claim
would be reproduced. The outcome rests on the windows taken for the control trains.

## figs2ef-control-stable

> In control mice without ICMS-driven behavioural learning, the number and spatial distribution of
> activated neurons stayed stable and resembled those of the early training weeks.

A claim of absence. The control files hold the binary activation of each ROI per channel and
current, so the count at 5 uA per channel and session is read directly; weeks 0 and 1 against
weeks 2 and 3, two-sided Mann-Whitney U (claims file, from the code). The density within 200 um
needs each ROI's distance to the stimulation contact: the files give ROI voxels in pixel and plane
indices with no pixel size or plane spacing, and contact positions by contact index 0 to 31 with
no mapping from the Ripple stimulation channel to the contact, so the density cannot be computed.
The behavioural mice's counts cannot be compared, as their volumetric activation cannot be derived
(sanity check above).

In [ ]:
x = CS[CS.current == 5.0].copy()
x["phase"] = np.where(x.week <= 1, "early", np.where(x.week <= 3, "late", None))
print("5 uA channel-sessions by mouse and week:")
print(x.groupby(["subject", "week"]).activated.agg(["size", "median"]).to_string())
e, l = x[x.phase == "early"].activated, x[x.phase == "late"].activated
p, r, nl, ne = mw(l, e)
print(f"count, weeks 0-1 median {e.median()} (n {ne}) against weeks 2-3 median {l.median()} (n {nl}): two-sided p = {p:.3g}, r = {r:.2f}")
fr = x.activated / x.n_rois
pf, rf, _, _ = mw(fr[x.phase == "late"], fr[x.phase == "early"])
print(f"variant, fraction of ROIs activated: {fr[x.phase == 'early'].median():.3f} vs {fr[x.phase == 'late'].median():.3f}, p = {pf:.3g}")
print("ROIs per session range %d to %d" % (x.n_rois.min(), x.n_rois.max()))
print("\nphases per mouse (channel-sessions early / late):",
      {m: (int((g.phase == "early").sum()), int((g.phase == "late").sum())) for m, g in x.groupby("subject")})
print("per mouse, early against late (mice with both phases):")
for m, g in x.groupby("subject"):
    ge, gl = g[g.phase == "early"].activated, g[g.phase == "late"].activated
    if len(ge) and len(gl):
        pm, rm, _, _ = mw(gl, ge)
        print(f"  {m}: medians {ge.median()} (n {len(ge)}) and {gl.median()} (n {len(gl)}), p = {pm:.3g}")
for name, keep in [("mice with both phases", x.subject.isin([m for m, g in x.groupby("subject") if g.phase.eq("early").any() and g.phase.eq("late").any()])),
                   ("sessions with at least 10 ROIs", x.n_rois >= 10),
                   ("both of these", x.subject.isin([m for m, g in x.groupby("subject") if g.phase.eq("early").any() and g.phase.eq("late").any()]) & (x.n_rois >= 10))]:
    y = x[keep]
    pv, rv, nlv, nev = mw(y[y.phase == "late"].activated, y[y.phase == "early"].activated)
    print(f"variant, {name}: medians {y[y.phase == 'early'].activated.median()} (n {nev}) and {y[y.phase == 'late'].activated.median()} (n {nlv}), p = {pv:.3g}")
# Ranks within mouse: each count's rank among the mouse's channel-sessions, so that a mouse seen in one phase only adds nothing to the contrast.
y = x[x.phase.notna()].copy()
y["rank"] = y.groupby("subject").activated.rank(pct=True)
pr, rr, nlr, ner = mw(y[y.phase == "late"]["rank"], y[y.phase == "early"]["rank"])
print(f"variant, within-mouse percentile ranks (mice with one phase add only ties at 0.5 or their spread): early median {y[y.phase == 'early']['rank'].median():.2f}, late {y[y.phase == 'late']['rank'].median():.2f}, p = {pr:.3g}")

**Result: not reproduced.** In the control mice the number of activated ROIs per channel-session
at 5 uA falls from a median of 12.5 in weeks 0 and 1 (n 54) to 2.0 in weeks 2 and 3 (n 60), p =
0.00378; as a fraction of ROIs, 0.063 against 0.027 (p = 0.00875). The count test finds the change
that the claim says is absent. The pooled test mixes mice: ICMS48 is seen only early (medians 67
and 46), and with the mice seen in both phases alone the medians are 4.0 and 2.0 (p = 0.444).
Within mice the counts are not stable either: they fall in ICMS45 (4.5 to 1.0, p = 0.00502) and
ICMS54 (8.0 to 1.0, p = 0.00628) and rise in ICMS56 (0.0 to 8.5, p = 0.00212). The density within
200 um could not be computed, and the comparison with the behavioural mice could not be made.

## Blocked claims

- fig2b-representative-recruitment, fig2c-more-activated-neurons, fig2d-density-peak-200um,
  fig2e-density-200um, fig2e-activation-radius, fig2g-at-threshold-stable (metadata_insufficient).
  The volumetric dF/F is one value per ROI and stored sample, with 6 to 8 samples per trial placed
  within 0.25 s of onset at the per-plane rate, and the value -1 at three to four of the eight
  positions in about half of the ROIs (sanity check). The files do not say which samples are
  pre-stimulus volumes and which stimulation volumes. The samples set to -1 sit, in the median over
  ROIs, at positions 3 and 7 on catch trials and 4 uA trials and at 0, 3 and 4 at 2, 3, 5 and 6 uA
  (one session), a layout the files do not explain. The activation of the Methods (voxels above twice their baseline STD) cannot
  be derived; the voxel-level data it needs are not in the files. The full description of the
  series says only "A1 footprint average of pre/mid 4D stacks; baseline = mean of pre frames"; read
  as four pre-stimulus and four stimulation samples, the mid minus pre difference is 0.114 on catch
  trials, against 0.080 to 0.320 at 2 to 6 uA with no order by current (sanity check), so that
  reading does not isolate an evoked response. The segmentation tables of all 45 behavioural files
  with imaging hold only ROI masks, sizes and source identifiers, with no activation or distance
  column. The behavioural files also give no position of the stimulation site in the imaging
  volume, which the distances and densities need.
- fig3b-tracking-counts, fig3df-stable-cells-no-change, fig3e-onset-decrease,
  fig3i-more-high-intensity, fig3j-subgroup-increase, fig3j-subgroup-divergence,
  fig3lm-spikes-and-latency, fig3n-new-vs-enhanced, fig3n-vs-week0 (metadata_insufficient). The
  planar ROIs are a separate segmentation in each session (53 to 1019 ROIs per session, sanity
  check), with no identifier that links them across sessions. The files hold no image of any
  kind: the segmentation tables' reference_images are empty in all 45 files with imaging,
  processing/ophys holds only the dF/F and the segmentations, and acquisition is empty, so the mean
  images the PatchWarp registration of the Methods works on are not in the files. figs6-imaging-pre-movement depends on fig3j and is
  blocked with it. Parts of fig3i (week 0 against week 4) and the recruitment of figs6 could be
  computed on all activated ROIs of each session without tracking, but the claims are about the
  same neurons followed over weeks (enhanced excitability of tracked neurons), and a comparison of
  separately segmented populations would test a different quantity; they are left blocked.
- fig4d-waveform-separation (data_missing). It needs waveforms of each unit's spikes in and out
  of stimulation; the files hold one mean waveform per unit on one channel.
- figs8-tracked-units (data_missing). UnitMatch needs each unit's multi-channel templates (from
  the sorting analyzers, Methods); the files hold one single-channel mean waveform per unit, and no
  tracking identity.

## Summary

In [ ]:
print(f"notebook run time {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

| Claim | Outcome |
|---|---|
| fig2a-threshold-decrease | reproduced (both readings) |
| figs1e-response-time | reproduced |
| figs5-movement-onset | reproduced, resting on assumptions |
| fig4h-unit-counts | not reproduced |
| figs3-modulation-all-cells | partially reproduced |
| fig4g-modulated-count-mouse1 | not reproduced, resting on an assumption |
| fig4ij-pyramidal-drive | partially reproduced, resting on an assumption |
| fig5d-pl-identity-stable | reproduced |
| fig5e-pyramidal-in-pl | partially reproduced |
| figs4b-conduction-velocity | not reproduced |
| fig5fg-pl-modulation-latency | partially reproduced (seed-sensitive) |
| figs4de-npl-no-change | reproduced (seed-sensitive) |
| fig5h-pl-spike-probability | partially reproduced |
| figs5-pre-movement-window | partially reproduced, resting on an assumption |
| figs7-control-no-change | not reproduced, resting on assumptions |
| figs2ef-control-stable | not reproduced |
| fig6c-kl-divergence | partially reproduced |
| fig6d-npl-hit-miss-divergence | not reproduced, resting on an assumption |
| fig6f-pl-coupling-increase | reproduced (seed-sensitive) |
| fig6f-npl-higher-coupling | reproduced |
| figs9-coupling-controls | partially reproduced, resting on an assumption |
| fig2b, fig2c, fig2d, fig2e (two), fig2g | blocked, metadata insufficient |
| fig3b, fig3df, fig3e, fig3i, fig3j (two), fig3lm, fig3n (two), figs6 | blocked, metadata insufficient |
| fig4d-waveform-separation, figs8-tracked-units | blocked, data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The pre-stimulus window is taken as 700 ms (the
Methods give none; the Results use 700 ms for the delta spikes); 300 and 500 ms change the outcome
of fig4g-modulated-count-mouse1. ICMS83 has no catch trials and no block structure in its files,
so its sessions are fitted as one block with a false-alarm rate of 0. The wheel's response
threshold is estimated from the data. Cell types come from the trough-to-peak at 0.45 ms recomputed from
the stored mean waveform, the criterion's rule, for every claim; the files' stored label is a
variant, under which fig4ij-pyramidal-drive would be reproduced. The hit and miss traces of Figure 6D are smoothed
per trial before the median. The control mice received 3 tabled bursts of 10 pulses at 50 Hz per
trial, and the windows used for them are assumptions; the spikes follow 10 bursts at 0.5 s, and
that reading changes the outcomes of figs7-control-no-change and figs9-coupling-controls. For the 120 ms pre-movement analysis
the responses are recomputed in the 120 ms windows. The paper's code reads processed tables that
are not public, so the response, pulse-locking and coupling measures are rebuilt from the Methods
and the code the claims file cites. Each claim draws its pulse-locking classes under its own seed.

What was found about the dataset. The volumetric dF/F has no documented relation between its
samples and the trial windows, and the segmentation tables have no activation or distance column;
the planar ROIs carry no cross-session identity, and the files hold no images to register them by; the stimulation site's position is missing from
the behavioural imaging files; the control imaging files give neither pixel size nor a mapping from
stimulation channel to probe contact. The control stimulation and trial tables list 3 bursts per
trial (30 pulses, 1.2 s) in most sessions, while the spikes respond every 0.5 s for 5 s; in ICMS45's
first ephys session the trial table gives 100 pulses over 4.7 s and the stimulation table 3 bursts.
ICMS83's files lack catch trials, and its wheel barely moves at the recorded response times. In
ICMS101's first session the trains start 56 to 85 ms after the trial start, against a median of
5.8 to 5.9 ms (5.5 to 37.4 ms) in the other sessions of five mice, and at the trial start in
ICMS83's. Some stimulation channels are missing from the electrodes table, and the data do not establish
that the stimulation table's channel numbers name the same contacts as the electrodes table's. 999 units are in the
files, against 973 in the paper. The sorted spike trains hold no spikes in the blanked first 1.5
ms after each pulse. Spikes and trial table share one clock in the behavioural sessions, as far
as the contrasts above show; the wheel moves more on hits than on misses in 51 of 54 sessions, the
other three being ICMS83's, whose wheel barely moves.